In [827]:
%pip install faiss-cpu
import faiss
import numpy as np
import re

from sentence_transformers import SentenceTransformer

print("FAISS version:", faiss.__version__)
print("NumPy version:", np.__version__)
print("Imports successful.")

Note: you may need to restart the kernel to use updated packages.
FAISS version: 1.15.1
NumPy version: 2.1.3
Imports successful.


In [828]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded:", MODEL_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2


In [829]:
from pypdf import PdfReader

PDF_PATH = "/kaggle/input/datasets/abdelrahmanisakr/llm-data6/LLM_DATASET.pdf"

reader = PdfReader(PDF_PATH)

full_text = "\n".join(
    page.extract_text() or ""
    for page in reader.pages
)

print("PDF pages:", len(reader.pages))
print("Extracted characters:", len(full_text))
print(full_text[:2000])

PDF pages: 22
Extracted characters: 26945
1. Junior Cybersecurity Engineer (0–2 Years):
Required Skills
Networking fundamentals: TCP/IP, DNS, HTTP/HTTPS, ports,
and firewalls.
Linux and Windows basics, command-line usage, and system
logs.
Security fundamentals: CIA triad, authentication, access control,
encryption, and common vulnerabilities.
Basic scripting with Python, Bash, or PowerShell.
Basic understanding of vulnerability assessment and incident
detection.
Preferred Skills
Familiarity with SIEM tools such as Splunk or Microsoft Sentinel.
Knowledge of OWASP Top 10 and common web vulnerabilities.
Basic cloud security knowledge.
Understanding of MITRE ATT&CK.
Education and Experience
Degree in Computer Science, Cybersecurity, IT, or a related field
is often preferred, but not universally required.
Personal labs, internships, CTFs, and documented projects can
demonstrate practical ability.
GPA requirements depend on the employer.
Recommended Certifications
CompTIA Security+.
ISC2 Cer

In [831]:
ROLES = [
    "Cybersecurity Engineer",
    "Robotics Engineer",
    "Generative AI Engineer",
    "Digital IC Design Engineer",
]

In [832]:
LEVELS = ["junior", "mid_level", "senior"]

SECTIONS = [
    "required_skills",
    "preferred_skills",
    "education_experience",
    "certifications",
    "projects",
    "cv_evidence",
]

In [833]:
import re

print("ROLES:", ROLES)
print("\nLEVELS:", LEVELS)
print("\nSECTIONS:", SECTIONS)

print("\n" + "=" * 60)
print("LINES THAT LOOK LIKE HEADINGS")
print("=" * 60)

for line in full_text.splitlines():
    line = line.strip()

    if not line:
        continue

    if (
        re.match(r"^\d+[\.\)]\s+", line)
        or line in {
            "Required Skills",
            "Preferred Skills",
            "Education and Experience",
            "Recommended Certifications",
            "Recommended Projects",
            "CV Evaluation Evidence",
        }
    ):
        print(repr(line))

ROLES: ['Cybersecurity Engineer', 'Robotics Engineer', 'Generative AI Engineer', 'Digital IC Design Engineer']

LEVELS: ['junior', 'mid_level', 'senior']

SECTIONS: ['required_skills', 'preferred_skills', 'education_experience', 'certifications', 'projects', 'cv_evidence']

LINES THAT LOOK LIKE HEADINGS
'1. Junior Cybersecurity Engineer (0–2 Years):'
'Required Skills'
'Preferred Skills'
'Education and Experience'
'Recommended Certifications'
'1) Cybersecurity:'
'Recommended Projects'
'CV Evaluation Evidence'
'2. Mid-Level Cybersecurity Engineer (2–5 Years)'
'Required Skills'
'Preferred Skills'
'Education and Experience'
'Recommended Certifications'
'Recommended Projects'
'CV Evaluation Evidence'
'3. Senior Cybersecurity Engineer (5+ Years)'
'Required Skills'
'Preferred Skills'
'Education and Experience'
'Recommended Certifications'
'Recommended Projects'
'CV Evaluation Evidence'
'2) Robotics Engineer:'
'1. Junior Robotics Engineer (0–2 Years)'
'Required Skills'
'Preferred Skills'
'Educ

In [834]:
import re

# Map PDF section headings to the keys used by the application.
SECTION_HEADINGS = {
    "Required Skills": "required_skills",
    "Preferred Skills": "preferred_skills",
    "Education and Experience": "education_experience",
    "Recommended Certifications": "certifications",
    "Recommended Projects": "projects",
    "CV Evaluation Evidence": "cv_evidence",
}

# Convert the level names in the PDF to our canonical level keys.
LEVEL_ALIASES = {
    "junior": "junior",
    "mid-level": "mid_level",
    "senior": "senior",
}

# Recognize the role-level headings in the PDF.
ROLE_LEVEL_PATTERN = re.compile(
    r"^\d+\.\s+"
    r"(Junior|Mid-Level|Senior)\s+"
    r"(Cybersecurity|Robotics|Generative AI|Digital IC Design)"
    r"\s+Engineer\b.*$",
    flags=re.IGNORECASE,
)

# Ignore page labels and extra role labels inserted by PDF extraction.
NOISE_PATTERN = re.compile(
    r"^(?:"
    r"Page\s+\d+"
    r"|Job Description Dataset"
    r"|\d+\)\s+(?:Cybersecurity|Robotics Engineer|"
    r"Generative AI Engineer|Digital IC Design Engineer):?"
    r")$",
    flags=re.IGNORECASE,
)


def parse_knowledge_base(text):
    parsed = []

    current_role = None
    current_level = None
    current_role_title = None
    current_section = None
    current_lines = []

    def save_section():
        """Save the current section if its role and heading are valid."""
        nonlocal current_lines

        if (
            current_role is None
            or current_level is None
            or current_section is None
        ):
            current_lines = []
            return

        cleaned_lines = []

        for line in current_lines:
            line = line.strip()

            if not line or NOISE_PATTERN.fullmatch(line):
                continue

            cleaned_lines.append(line)

        # Join PDF-wrapped lines into readable text.
        section_text = " ".join(cleaned_lines)
        section_text = re.sub(r"\s+", " ", section_text).strip()

        parsed.append({
            "role": current_role,
            "level": current_level,
            "role_title": current_role_title,
            "section": current_section,
            "title": current_section.replace("_", " ").title(),
            "text": section_text,
        })

        current_lines = []

    for raw_line in text.splitlines():
        line = raw_line.strip()

        if not line:
            continue

        # Start a new role-level record group.
        match = ROLE_LEVEL_PATTERN.match(line)

        if match:
            save_section()

            level_label = match.group(1).lower()
            role_name = match.group(2).lower()

            current_level = LEVEL_ALIASES[level_label]

            # Match the extracted role name to the canonical role list.
            current_role = next(
                (
                    role for role in ROLES
                    if role.lower().startswith(role_name)
                ),
                None,
            )

            if current_role is None:
                raise ValueError(
                    f"Could not map role heading: {line}"
                )

            display_level = {
                "junior": "Junior",
                "mid_level": "Mid-Level",
                "senior": "Senior",
            }[current_level]

            current_role_title = f"{display_level} {current_role}"
            current_section = None
            current_lines = []
            continue

        # Start a new section within the current role and level.
        if line in SECTION_HEADINGS:
            save_section()
            current_section = SECTION_HEADINGS[line]
            current_lines = []
            continue

        # Skip unrelated document headings and page labels.
        if NOISE_PATTERN.fullmatch(line):
            continue

        # Collect text only after a recognized section heading.
        if current_role is not None and current_section is not None:
            current_lines.append(line)

    # Save the final section at the end of the PDF.
    save_section()

    return parsed


# Create the missing knowledge-base chunks.
chunks = parse_knowledge_base(full_text)

print("Parser restored successfully.")
print("Total parsed sections:", len(chunks))
print("\nFirst parsed record:")
print(chunks[0])

Parser restored successfully.
Total parsed sections: 72

First parsed record:
{'role': 'Cybersecurity Engineer', 'level': 'junior', 'role_title': 'Junior Cybersecurity Engineer', 'section': 'required_skills', 'title': 'Required Skills', 'text': 'Networking fundamentals: TCP/IP, DNS, HTTP/HTTPS, ports, and firewalls. Linux and Windows basics, command-line usage, and system logs. Security fundamentals: CIA triad, authentication, access control, encryption, and common vulnerabilities. Basic scripting with Python, Bash, or PowerShell. Basic understanding of vulnerability assessment and incident detection.'}


In [836]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(MODEL_NAME)

documents = [item["text"] for item in chunks]

embeddings = embedding_model.encode(
    documents,
    batch_size=16,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

embeddings = np.asarray(embeddings, dtype=np.float32)

assert embeddings.shape == (72, 384), (
    f"Unexpected embedding shape: {embeddings.shape}"
)
assert np.isfinite(embeddings).all(), "Embeddings contain invalid values"

embedding_dimension = embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dimension)
faiss_index.add(embeddings)

assert faiss_index.ntotal == len(chunks), (
    "FAISS index size does not match the knowledge base"
)

print("\nEmbeddings shape:", embeddings.shape)
print("FAISS vectors:", faiss_index.ntotal)
print("Embedding dimension:", embedding_dimension)
print("PASS: Knowledge base indexed successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]


Embeddings shape: (72, 384)
FAISS vectors: 72
Embedding dimension: 384
PASS: Knowledge base indexed successfully


In [837]:
def retrieve_sections(
    query,
    role,
    level,
    top_k=3,
    allowed_sections=None,
):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    # Search the complete index, then filter by role and level.
    scores, indices = faiss_index.search(
        query_embedding,
        faiss_index.ntotal,
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        item = chunks[int(idx)]

        if item["role"] != role or item["level"] != level:
            continue

        if (
            allowed_sections is not None
            and item["section"] not in allowed_sections
        ):
            continue

        results.append({
            "role": item["role"],
            "level": item["level"],
            "section": item["section"],
            "score": float(score),
            "text": item["text"],
        })

        if len(results) >= top_k:
            break

    return results

In [838]:
results = retrieve_sections(
    query=(
        "What evidence should a senior digital IC design engineer "
        "show for architectural ownership, timing closure, and PPA?"
    ),
    role="Digital IC Design Engineer",
    level="senior",
    top_k=3,
)

assert results, "No results returned."
assert all(
    item["role"] == "Digital IC Design Engineer"
    and item["level"] == "senior"
    for item in results
), "Role or level filtering failed."

for i, item in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Section:", item["section"])
    print("Similarity:", round(item["score"], 4))
    print("Text:", item["text"])

print("\nPASS: Retrieval respects role and level.")


Result 1
Section: cv_evidence
Similarity: 0.62
Text: Look for architectural ownership, successful subsystem delivery, measurable PPA results, timing-closure expertise, integration leadership, design-quality improvements, and collaboration across implementation and verification teams.

Result 2
Section: education_experience
Similarity: 0.6101
Text: Significant relevant experience and evidence of technical ownership are generally expected. A bachelor's degree is common; a master's or PhD may be preferred for research-intensive or specialized positions but is not universally required. Evidence should include architectural decisions, subsystem delivery, successful timing closure, PPA improvements, or silicon- debug contributions. Seniority should reflect demonstrated scope and impact, not years alone.

Result 3
Section: certifications
Similarity: 0.4408
Text: Advanced EDA-vendor or ASIC design-flow training where relevant. Specialized training in low-power design, formal methods, timing c

In [839]:
SECTION_ROUTING = {
    "skills": [
        "required_skills",
        "preferred_skills",
    ],
    "projects": [
        "projects",
    ],
    "certifications": [
        "certifications",
    ],
    "education": [
        "education_experience",
    ],
    "cv_evidence": [
        "cv_evidence",
    ],
}


def retrieve_for_question(
    question,
    role,
    level,
    question_type,
    top_k=3,
):
    if question_type not in SECTION_ROUTING:
        raise ValueError(
            f"Unknown question type: {question_type}. "
            f"Choose from: {list(SECTION_ROUTING.keys())}"
        )

    return retrieve_sections(
        query=question,
        role=role,
        level=level,
        top_k=top_k,
        allowed_sections=SECTION_ROUTING[question_type],
    )


print("Section routing restored.")
print("Supported question types:", list(SECTION_ROUTING.keys()))

Section routing restored.
Supported question types: ['skills', 'projects', 'certifications', 'education', 'cv_evidence']


In [840]:
results = retrieve_for_question(
    question=(
        "What evidence demonstrates senior-level architectural ownership, "
        "PPA optimization, and timing closure?"
    ),
    role="Digital IC Design Engineer",
    level="senior",
    question_type="cv_evidence",
    top_k=3,
)

assert results, "No results returned."
assert all(
    item["section"] == "cv_evidence"
    for item in results
), "Section routing returned an unexpected section."

for i, item in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Section:", item["section"])
    print("Similarity:", round(item["score"], 4))
    print("Text:", item["text"])

print("\nPASS: Section routing works.")


Result 1
Section: cv_evidence
Similarity: 0.6344
Text: Look for architectural ownership, successful subsystem delivery, measurable PPA results, timing-closure expertise, integration leadership, design-quality improvements, and collaboration across implementation and verification teams.

PASS: Section routing works.


In [841]:
EVALUATION_SECTIONS = [
    "required_skills",
    "preferred_skills",
    "cv_evidence",
]


def get_evaluation_criteria(role, level):
    criteria = []

    for item in chunks:
        if item["role"] != role or item["level"] != level:
            continue

        if item["section"] not in EVALUATION_SECTIONS:
            continue

        criteria.append({
            "role": item["role"],
            "level": item["level"],
            "section": item["section"],
            "title": item["title"],
            "criteria_text": item["text"],
        })

    return criteria


robotics_criteria = get_evaluation_criteria(
    role="Robotics Engineer",
    level="mid_level",
)

print("Criteria sections found:", len(robotics_criteria))

for item in robotics_criteria:
    print(f"\nSection: {item['section']}")
    print("Text:", item["criteria_text"])

Criteria sections found: 3

Section: required_skills
Text: Strong Python and/or C++ development for robotics applications. Practical ROS 2 development, including communication, lifecycle management, launch systems, and debugging. Kinematics, dynamics, feedback control, and trajectory generation appropriate to the robot. Sensor integration and coordinate-frame management using TF2. Localization, mapping, navigation, or manipulation according to specialization. Software testing, performance profiling, Git workflows, and integration of hardware and software. Ability to diagnose failures across sensors, algorithms, middleware, and robot hardware.

Section: preferred_skills
Text: SLAM, visual odometry, sensor fusion, or state estimation. Motion planning using MoveIt 2, Nav2, or relevant planning libraries. Real-time systems, embedded Linux, microcontrollers, or firmware integration. Computer vision and deep learning for robotic perception. Simulation-to-real transfer and automated testing. 

In [842]:
import re


def split_into_criteria(criteria_sections):
    individual_criteria = []

    for section in criteria_sections:
        text = section["criteria_text"].strip()

        # Repair words split across PDF line breaks.
        text = re.sub(r"-\s*\n\s*", "-", text)
        text = re.sub(r"(?<=[a-z,;])\n(?=[a-z])", " ", text)

        # Split at sentence boundaries.
        sentences = re.split(
            r"(?<=[.!?])\s+(?=[A-Z0-9])",
            text,
        )

        for sentence in sentences:
            criterion = sentence.strip().lstrip("-•* ").strip()

            if criterion:
                individual_criteria.append({
                    "role": section["role"],
                    "level": section["level"],
                    "source_section": section["section"],
                    "criterion": criterion,
                })

    return individual_criteria


robotics_individual_criteria = split_into_criteria(
    robotics_criteria
)

print("Individual criteria:", len(robotics_individual_criteria))

for i, item in enumerate(robotics_individual_criteria, start=1):
    print(
        f"{i}. [{item['source_section']}] "
        f"{item['criterion']}"
    )

Individual criteria: 14
1. [required_skills] Strong Python and/or C++ development for robotics applications.
2. [required_skills] Practical ROS 2 development, including communication, lifecycle management, launch systems, and debugging.
3. [required_skills] Kinematics, dynamics, feedback control, and trajectory generation appropriate to the robot.
4. [required_skills] Sensor integration and coordinate-frame management using TF2.
5. [required_skills] Localization, mapping, navigation, or manipulation according to specialization.
6. [required_skills] Software testing, performance profiling, Git workflows, and integration of hardware and software.
7. [required_skills] Ability to diagnose failures across sensors, algorithms, middleware, and robot hardware.
8. [preferred_skills] SLAM, visual odometry, sensor fusion, or state estimation.
9. [preferred_skills] Motion planning using MoveIt 2, Nav2, or relevant planning libraries.
10. [preferred_skills] Real-time systems, embedded Linux, microc

In [843]:
ATOMIC_CRITERIA_MAP = {
    "Strong Python and/or C++ development for robotics applications.": [
        "Python development for robotics applications",
        "C++ development for robotics applications",
    ],
    (
        "Practical ROS 2 development, including communication, "
        "lifecycle management, launch systems, and debugging."
    ): [
        "ROS 2 application development",
        "ROS 2 communication",
        "ROS 2 lifecycle management",
        "ROS 2 launch systems",
        "ROS 2 debugging",
    ],
    (
        "Kinematics, dynamics, feedback control, and trajectory "
        "generation appropriate to the robot."
    ): [
        "Robot kinematics",
        "Robot dynamics",
        "Feedback control",
        "Trajectory generation",
    ],
    "Sensor integration and coordinate-frame management using TF2.": [
        "Sensor integration",
        "Coordinate-frame management using TF2",
    ],
    (
        "Localization, mapping, navigation, or manipulation "
        "according to specialization."
    ): [
        "Localization",
        "Mapping",
        "Navigation",
        "Manipulation",
    ],
    (
        "Software testing, performance profiling, Git workflows, "
        "and integration of hardware and software."
    ): [
        "Software testing",
        "Performance profiling",
        "Git workflows",
        "Hardware/software integration",
    ],
    (
        "Ability to diagnose failures across sensors, algorithms, "
        "middleware, and robot hardware."
    ): [
        "Sensor failure diagnosis",
        "Algorithm failure diagnosis",
        "Middleware failure diagnosis",
        "Robot hardware failure diagnosis",
    ],
    "SLAM, visual odometry, sensor fusion, or state estimation.": [
        "SLAM",
        "Visual odometry",
        "Sensor fusion",
        "State estimation",
    ],
    (
        "Motion planning using MoveIt 2, Nav2, or relevant planning libraries."
    ): [
        "Motion planning",
        "Nav2",
        "MoveIt 2",
    ],
    (
        "Real-time systems, embedded Linux, microcontrollers, "
        "or firmware integration."
    ): [
        "Real-time systems",
        "Embedded Linux",
        "Microcontrollers",
        "Firmware integration",
    ],
    "Computer vision and deep learning for robotic perception.": [
        "Computer vision for robotics",
        "Deep learning for robotic perception",
    ],
    "Simulation-to-real transfer and automated testing.": [
        "Simulation-to-real transfer",
        "Automated testing",
    ],
    (
        "Containerization, CI/CD, and deployment on robotic compute platforms."
    ): [
        "Containerization",
        "CI/CD",
        "Deployment on robotic compute platforms",
    ],
}


atomic_criteria = []

for item in robotics_individual_criteria:
    original = item["criterion"]

    # The CV evidence guideline describes how to evaluate evidence;
    # it is not an additional technical skill to score.
    if item["source_section"] == "cv_evidence":
        continue

    atomic_names = ATOMIC_CRITERIA_MAP.get(original, [original])

    for atomic_name in atomic_names:
        atomic_criteria.append({
            "role": item["role"],
            "level": item["level"],
            "source_section": item["source_section"],
            "original_criterion": original,
            "criterion": atomic_name,
        })


print("Atomic technical criteria:", len(atomic_criteria))

for i, item in enumerate(atomic_criteria, start=1):
    print(
        f"{i}. [{item['source_section']}] "
        f"{item['criterion']}"
    )

Atomic technical criteria: 43
1. [required_skills] Python development for robotics applications
2. [required_skills] C++ development for robotics applications
3. [required_skills] ROS 2 application development
4. [required_skills] ROS 2 communication
5. [required_skills] ROS 2 lifecycle management
6. [required_skills] ROS 2 launch systems
7. [required_skills] ROS 2 debugging
8. [required_skills] Robot kinematics
9. [required_skills] Robot dynamics
10. [required_skills] Feedback control
11. [required_skills] Trajectory generation
12. [required_skills] Sensor integration
13. [required_skills] Coordinate-frame management using TF2
14. [required_skills] Localization
15. [required_skills] Mapping
16. [required_skills] Navigation
17. [required_skills] Manipulation
18. [required_skills] Software testing
19. [required_skills] Performance profiling
20. [required_skills] Git workflows
21. [required_skills] Hardware/software integration
22. [required_skills] Sensor failure diagnosis
23. [required

In [844]:
ALTERNATIVE_GROUPS = {
    "programming_languages": {
        "source_section": "required_skills",
        "mode": "any",
        "criteria": [
            "Python development for robotics applications",
            "C++ development for robotics applications",
        ],
        "reason": "The original requirement says Python and/or C++.",
    },

    "robot_specialization": {
        "source_section": "required_skills",
        "mode": "any",
        "criteria": [
            "Localization",
            "Mapping",
            "Navigation",
            "Manipulation",
        ],
        "reason": (
            "The original requirement says these depend on "
            "the candidate's robotics specialization."
        ),
    },

    "state_estimation_methods": {
        "source_section": "preferred_skills",
        "mode": "any",
        "criteria": [
            "SLAM",
            "Visual odometry",
            "Sensor fusion",
            "State estimation",
        ],
        "reason": "The original requirement lists alternative methods.",
    },
}

# Validate that every alternative refers to an existing criterion.
atomic_names = {item["criterion"] for item in atomic_criteria}

for group_name, group in ALTERNATIVE_GROUPS.items():
    for criterion in group["criteria"]:
        assert criterion in atomic_names, (
            f"Unknown criterion in {group_name}: {criterion}"
        )

    assert group["mode"] == "any"

print("Alternative-group validation passed.")
print("Atomic technical criteria:", len(atomic_criteria))

for group_name, group in ALTERNATIVE_GROUPS.items():
    print(f"\n{group_name}")
    print("Rule:", group["mode"])
    print("Reason:", group["reason"])

    for criterion in group["criteria"]:
        print(" -", criterion)

Alternative-group validation passed.
Atomic technical criteria: 43

programming_languages
Rule: any
Reason: The original requirement says Python and/or C++.
 - Python development for robotics applications
 - C++ development for robotics applications

robot_specialization
Rule: any
Reason: The original requirement says these depend on the candidate's robotics specialization.
 - Localization
 - Mapping
 - Navigation
 - Manipulation

state_estimation_methods
Rule: any
Reason: The original requirement lists alternative methods.
 - SLAM
 - Visual odometry
 - Sensor fusion
 - State estimation


In [845]:
import re


def split_cv_sentences(cv_text):
    lines = [
        line.strip(" -\t")
        for line in cv_text.splitlines()
        if line.strip()
    ]

    sentences = []

    for line in lines:
        parts = re.split(r"(?<=[.!?])\s+", line)
        sentences.extend(
            part.strip()
            for part in parts
            if part.strip()
        )

    return sentences


def extract_cv_evidence_sentences(cv_text):
    sentences = split_cv_sentences(cv_text)

    cleaned_sentences = []

    for sentence in sentences:
        # Ignore common CV headings, not evidence-bearing statements.
        if re.match(
            r"^(robotics engineer|curriculum vitae|resume|cv)\s*[—–:-]",
            sentence,
            flags=re.IGNORECASE,
        ):
            continue

        cleaned_sentences.append(sentence)

    return cleaned_sentences


sample_cv = """
Robotics Engineer — Mid-Level Candidate

- Developed ROS 2 applications for an autonomous mobile robot.
- Integrated Nav2 navigation and tested SLAM in Gazebo simulation.
- Combined IMU and camera data for sensor fusion.
- Measured localization error and navigation success rate.
- No physical robot deployment is described in this CV excerpt.
"""

cv_sentences = extract_cv_evidence_sentences(sample_cv)

print("Extracted CV statements:", len(cv_sentences))

for i, sentence in enumerate(cv_sentences, start=1):
    print(f"{i}. {sentence}")

Extracted CV statements: 5
1. Developed ROS 2 applications for an autonomous mobile robot.
2. Integrated Nav2 navigation and tested SLAM in Gazebo simulation.
3. Combined IMU and camera data for sensor fusion.
4. Measured localization error and navigation success rate.
5. No physical robot deployment is described in this CV excerpt.


In [846]:
NEGATION_PATTERNS = [
    r"\bno\b",
    r"\bnot\b",
    r"\bnever\b",
    r"\bwithout\b",
    r"\bunable to\b",
    r"\black(?:s|ed|ing)?\b",
    r"\bhasn't\b",
    r"\bhaven't\b",
    r"\bdidn't\b",
    r"\bdoesn't\b",
]


def contains_explicit_negation(text):
    text_lower = text.lower()

    return any(
        re.search(pattern, text_lower)
        for pattern in NEGATION_PATTERNS
    )


def classify_evidence(sentence, related_skill_mentioned=True):
    """
    Baseline classifier.

    This checks for explicit negation only.
    It does NOT establish whether the candidate truly has a skill.
    """
    if not related_skill_mentioned:
        return "not_mentioned"

    if contains_explicit_negation(sentence):
        return "explicitly_contradicted"

    return "potential_positive_evidence"


test_cases = [
    (
        "Developed ROS 2 applications for an autonomous robot.",
        True,
        "potential_positive_evidence",
    ),
    (
        "Integrated IMU and camera data for sensor fusion.",
        True,
        "potential_positive_evidence",
    ),
    (
        "No physical robot deployment is described in this CV excerpt.",
        True,
        "explicitly_contradicted",
    ),
    (
        "Worked on robotics software.",
        False,
        "not_mentioned",
    ),
]

for sentence, mentioned, expected in test_cases:
    actual = classify_evidence(sentence, mentioned)

    print("Statement:", sentence)
    print("Expected:", expected)
    print("Actual:", actual)
    print("PASS" if actual == expected else "FAIL")

Statement: Developed ROS 2 applications for an autonomous robot.
Expected: potential_positive_evidence
Actual: potential_positive_evidence
PASS
Statement: Integrated IMU and camera data for sensor fusion.
Expected: potential_positive_evidence
Actual: potential_positive_evidence
PASS
Statement: No physical robot deployment is described in this CV excerpt.
Expected: explicitly_contradicted
Actual: explicitly_contradicted
PASS
Statement: Worked on robotics software.
Expected: not_mentioned
Actual: not_mentioned
PASS


In [847]:
def classify_evidence_statement(sentence):
    text = sentence.lower().strip()

    # These phrases indicate missing documentation, not missing ability.
    documentation_absence_patterns = [
        r"\bnot (?:described|mentioned|documented|specified)\b",
        r"\bno (?:evidence|information|details)\b",
        r"\bnot included in (?:this )?(?:cv|resume|excerpt)\b",
        r"\bno .{1,80}\b(?:is|was|are|were) "
        r"(?:described|mentioned|documented|specified)\b",
    ]

    if any(
        re.search(pattern, text)
        for pattern in documentation_absence_patterns
    ):
        return "not_mentioned"

    # Direct negative claims about the candidate's experience or ability.
    explicit_negative_patterns = [
        r"\bnever (?:used|developed|implemented|performed|worked with)\b",
        r"\bno experience (?:with|in)\b",
        r"\bhas no experience\b",
        r"\bhas never\b",
        r"\bhas not used\b",
        r"\bhas not performed\b",
        r"\bunable to\b",
        r"\blacks? experience\b",
    ]

    if any(
        re.search(pattern, text)
        for pattern in explicit_negative_patterns
    ):
        return "explicitly_contradicted"

    # Action verbs trigger human review rather than an automatic skill verdict.
    positive_action_patterns = [
        r"\bdeveloped\b",
        r"\bimplemented\b",
        r"\bintegrated\b",
        r"\bdesigned\b",
        r"\btested\b",
        r"\bmeasured\b",
        r"\boptimized\b",
        r"\bdeployed\b",
        r"\bdebugged\b",
    ]

    if any(
        re.search(pattern, text)
        for pattern in positive_action_patterns
    ):
        return "needs_review"

    return "needs_review"


test_cases = [
    (
        "Developed ROS 2 applications for an autonomous robot.",
        "needs_review",
    ),
    (
        "No physical robot deployment is described in this CV excerpt.",
        "not_mentioned",
    ),
    (
        "The candidate has no experience with physical robot deployment.",
        "explicitly_contradicted",
    ),
    (
        "Worked on robotics software.",
        "needs_review",
    ),
]

for sentence, expected in test_cases:
    actual = classify_evidence_statement(sentence)

    print(f"\nStatement: {sentence}")
    print(f"Expected:  {expected}")
    print(f"Actual:    {actual}")

    assert actual == expected, (
        f"Classification mismatch: {sentence}"
    )

print("\nPASS: All evidence-classification tests passed.")


Statement: Developed ROS 2 applications for an autonomous robot.
Expected:  needs_review
Actual:    needs_review

Statement: No physical robot deployment is described in this CV excerpt.
Expected:  not_mentioned
Actual:    not_mentioned

Statement: The candidate has no experience with physical robot deployment.
Expected:  explicitly_contradicted
Actual:    explicitly_contradicted

Statement: Worked on robotics software.
Expected:  needs_review
Actual:    needs_review

PASS: All evidence-classification tests passed.


In [848]:

def retrieve_relevant_criteria(
    evidence_sentence,
    criteria,
    top_k=3,
):
    query_embedding = embedding_model.encode(
        [evidence_sentence],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    criterion_texts = [
        item["criterion"]
        for item in criteria
    ]

    criterion_embeddings = embedding_model.encode(
        criterion_texts,
        batch_size=16,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)

    similarities = criterion_embeddings @ query_embedding[0]

    ranked_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in ranked_indices:
        results.append({
            "criterion": criteria[int(idx)]["criterion"],
            "source_section": criteria[int(idx)]["source_section"],
            "similarity": float(similarities[int(idx)]),
        })

    return results


test_evidence = (
    "Integrated Nav2 navigation and tested SLAM in Gazebo simulation."
)

matches = retrieve_relevant_criteria(
    evidence_sentence=test_evidence,
    criteria=atomic_criteria,
    top_k=5,
)

for rank, item in enumerate(matches, start=1):
    print(f"\n{rank}. {item['criterion']}")
    print(f"   Section: {item['source_section']}")
    print(f"   Similarity: {item['similarity']:.4f}")



1. Navigation
   Section: required_skills
   Similarity: 0.5474

2. Nav2
   Section: preferred_skills
   Similarity: 0.4886

3. SLAM
   Section: preferred_skills
   Similarity: 0.4242

4. Visual odometry
   Section: preferred_skills
   Similarity: 0.4048

5. ROS 2 launch systems
   Section: required_skills
   Similarity: 0.3285


In [849]:

cv_sentences = extract_cv_evidence_sentences(sample_cv)

for sentence in cv_sentences:
    print("\n" + "=" * 80)
    print(f"CV EVIDENCE: {sentence}")

    matches = retrieve_relevant_criteria(
        evidence_sentence=sentence,
        criteria=atomic_criteria,
        top_k=3,
    )

    for rank, item in enumerate(matches, start=1):
        print(
            f"{rank}. {item['criterion']} "
            f"[{item['source_section']}] "
            f"(similarity={item['similarity']:.4f})"
        )



CV EVIDENCE: Developed ROS 2 applications for an autonomous mobile robot.
1. ROS 2 application development [required_skills] (similarity=0.6952)
2. ROS 2 launch systems [required_skills] (similarity=0.6393)
3. ROS 2 communication [required_skills] (similarity=0.6290)

CV EVIDENCE: Integrated Nav2 navigation and tested SLAM in Gazebo simulation.
1. Navigation [required_skills] (similarity=0.5474)
2. Nav2 [preferred_skills] (similarity=0.4886)
3. SLAM [preferred_skills] (similarity=0.4242)

CV EVIDENCE: Combined IMU and camera data for sensor fusion.
1. Sensor fusion [preferred_skills] (similarity=0.7298)
2. Sensor integration [required_skills] (similarity=0.4703)
3. Visual odometry [preferred_skills] (similarity=0.3896)

CV EVIDENCE: Measured localization error and navigation success rate.
1. Localization [required_skills] (similarity=0.5667)
2. Visual odometry [preferred_skills] (similarity=0.4517)
3. Navigation [required_skills] (similarity=0.4482)

CV EVIDENCE: No physical robot dep

In [850]:

def analyze_cv_evidence(cv_text, criteria, top_k=3):
    sentences = extract_cv_evidence_sentences(cv_text)
    report = []

    for sentence in sentences:
        classification = classify_evidence_statement(sentence)

        matches = retrieve_relevant_criteria(
            evidence_sentence=sentence,
            criteria=criteria,
            top_k=top_k,
        )

        report.append({
            "evidence": sentence,
            "classification": classification,
            "matches": matches,
        })

    return report


evidence_report = analyze_cv_evidence(
    cv_text=sample_cv,
    criteria=atomic_criteria,
    top_k=3,
)

for item in evidence_report:
    print("\n" + "=" * 80)
    print("Evidence:", item["evidence"])
    print("Classification:", item["classification"])
    print("Matched criteria:")

    for match in item["matches"]:
        print(
            f"  - {match['criterion']} "
            f"[{match['source_section']}] "
            f"(similarity={match['similarity']:.4f})"
        )



Evidence: Developed ROS 2 applications for an autonomous mobile robot.
Classification: needs_review
Matched criteria:
  - ROS 2 application development [required_skills] (similarity=0.6952)
  - ROS 2 launch systems [required_skills] (similarity=0.6393)
  - ROS 2 communication [required_skills] (similarity=0.6290)

Evidence: Integrated Nav2 navigation and tested SLAM in Gazebo simulation.
Classification: needs_review
Matched criteria:
  - Navigation [required_skills] (similarity=0.5474)
  - Nav2 [preferred_skills] (similarity=0.4886)
  - SLAM [preferred_skills] (similarity=0.4242)

Evidence: Combined IMU and camera data for sensor fusion.
Classification: needs_review
Matched criteria:
  - Sensor fusion [preferred_skills] (similarity=0.7298)
  - Sensor integration [required_skills] (similarity=0.4703)
  - Visual odometry [preferred_skills] (similarity=0.3896)

Evidence: Measured localization error and navigation success rate.
Classification: needs_review
Matched criteria:
  - Localizati

In [851]:

def label_similarity(similarity):
    if similarity >= 0.60:
        return "strong_match"
    elif similarity >= 0.40:
        return "possible_match"
    else:
        return "weak_match"


for item in evidence_report:
    print("\n" + "=" * 80)
    print("Evidence:", item["evidence"])
    print("Evidence status:", item["classification"])

    for match in item["matches"]:
        label = label_similarity(match["similarity"])

        print(
            f"- {match['criterion']} "
            f"| {match['source_section']} "
            f"| similarity={match['similarity']:.4f} "
            f"| {label}"
        )



Evidence: Developed ROS 2 applications for an autonomous mobile robot.
Evidence status: needs_review
- ROS 2 application development | required_skills | similarity=0.6952 | strong_match
- ROS 2 launch systems | required_skills | similarity=0.6393 | strong_match
- ROS 2 communication | required_skills | similarity=0.6290 | strong_match

Evidence: Integrated Nav2 navigation and tested SLAM in Gazebo simulation.
Evidence status: needs_review
- Navigation | required_skills | similarity=0.5474 | possible_match
- Nav2 | preferred_skills | similarity=0.4886 | possible_match
- SLAM | preferred_skills | similarity=0.4242 | possible_match

Evidence: Combined IMU and camera data for sensor fusion.
Evidence status: needs_review
- Sensor fusion | preferred_skills | similarity=0.7298 | strong_match
- Sensor integration | required_skills | similarity=0.4703 | possible_match
- Visual odometry | preferred_skills | similarity=0.3896 | weak_match

Evidence: Measured localization error and navigation suc

In [852]:

def filter_candidate_matches(
    evidence_item,
    minimum_similarity=0.40,
):
    # Missing documentation is not evidence of skill possession.
    if evidence_item["classification"] == "not_mentioned":
        return []

    return [
        match
        for match in evidence_item["matches"]
        if match["similarity"] >= minimum_similarity
    ]


for item in evidence_report:
    candidate_matches = filter_candidate_matches(item)

    print("\n" + "=" * 80)
    print("Evidence:", item["evidence"])
    print("Evidence status:", item["classification"])

    if not candidate_matches:
        print("Candidate matches: none")
        continue

    print("Candidate matches for review:")
    for match in candidate_matches:
        print(
            f"- {match['criterion']} "
            f"[{match['source_section']}] "
            f"(similarity={match['similarity']:.4f})"
        )



Evidence: Developed ROS 2 applications for an autonomous mobile robot.
Evidence status: needs_review
Candidate matches for review:
- ROS 2 application development [required_skills] (similarity=0.6952)
- ROS 2 launch systems [required_skills] (similarity=0.6393)
- ROS 2 communication [required_skills] (similarity=0.6290)

Evidence: Integrated Nav2 navigation and tested SLAM in Gazebo simulation.
Evidence status: needs_review
Candidate matches for review:
- Navigation [required_skills] (similarity=0.5474)
- Nav2 [preferred_skills] (similarity=0.4886)
- SLAM [preferred_skills] (similarity=0.4242)

Evidence: Combined IMU and camera data for sensor fusion.
Evidence status: needs_review
Candidate matches for review:
- Sensor fusion [preferred_skills] (similarity=0.7298)
- Sensor integration [required_skills] (similarity=0.4703)

Evidence: Measured localization error and navigation success rate.
Evidence status: needs_review
Candidate matches for review:
- Localization [required_skills] (sim

In [853]:

def build_criterion_evidence_table(
    evidence_report,
    criteria,
    minimum_similarity=0.40,
):
    table = []

    for criterion_item in criteria:
        criterion = criterion_item["criterion"]
        candidates = []

        for evidence_item in evidence_report:
            # Do not treat missing documentation as positive evidence.
            if evidence_item["classification"] == "not_mentioned":
                continue

            for match in evidence_item["matches"]:
                if match["criterion"] != criterion:
                    continue

                if match["similarity"] < minimum_similarity:
                    continue

                candidates.append({
                    "evidence": evidence_item["evidence"],
                    "similarity": match["similarity"],
                    "classification": evidence_item["classification"],
                })

        candidates.sort(
            key=lambda candidate: candidate["similarity"],
            reverse=True,
        )

        table.append({
            "criterion": criterion,
            "source_section": criterion_item["source_section"],
            "best_evidence": (
                candidates[0]["evidence"] if candidates else None
            ),
            "similarity": (
                candidates[0]["similarity"] if candidates else None
            ),
            "candidate_count": len(candidates),
        })

    return table


criterion_evidence_table = build_criterion_evidence_table(
    evidence_report=evidence_report,
    criteria=atomic_criteria,
)

for item in criterion_evidence_table:
    if item["best_evidence"] is not None:
        print(f"\nCriterion: {item['criterion']}")
        print(f"Section: {item['source_section']}")
        print(f"Best evidence: {item['best_evidence']}")
        print(f"Similarity: {item['similarity']:.4f}")
        print(f"Matching statements: {item['candidate_count']}")



Criterion: ROS 2 application development
Section: required_skills
Best evidence: Developed ROS 2 applications for an autonomous mobile robot.
Similarity: 0.6952
Matching statements: 1

Criterion: ROS 2 communication
Section: required_skills
Best evidence: Developed ROS 2 applications for an autonomous mobile robot.
Similarity: 0.6290
Matching statements: 1

Criterion: ROS 2 launch systems
Section: required_skills
Best evidence: Developed ROS 2 applications for an autonomous mobile robot.
Similarity: 0.6393
Matching statements: 1

Criterion: Sensor integration
Section: required_skills
Best evidence: Combined IMU and camera data for sensor fusion.
Similarity: 0.4703
Matching statements: 1

Criterion: Localization
Section: required_skills
Best evidence: Measured localization error and navigation success rate.
Similarity: 0.5667
Matching statements: 1

Criterion: Navigation
Section: required_skills
Best evidence: Integrated Nav2 navigation and tested SLAM in Gazebo simulation.
Similarity:

In [854]:

CRITERION_KEYWORDS = {
    "ROS 2 application development": [
        "ros 2", "ros2", "robot operating system",
    ],
    "ROS 2 communication": [
        "topic", "service", "action", "publisher", "subscriber",
        "ros 2 communication", "ros2 communication",
    ],
    "ROS 2 launch systems": [
        "launch file", "launch system", "ros2 launch", "ros 2 launch",
    ],
    "Sensor integration": [
        "sensor integration", "integrated imu", "integrated camera",
        "integrated sensors",
    ],
    "Localization": [
        "localization", "localisation", "localization error",
    ],
    "Navigation": [
        "navigation", "nav2",
    ],
    "SLAM": [
        "slam", "simultaneous localization and mapping",
    ],
    "Visual odometry": [
        "visual odometry", "vo pipeline",
    ],
    "Sensor fusion": [
        "sensor fusion", "fused imu", "fusion of imu",
        "combined imu and camera data",
    ],
    "Nav2": [
        "nav2",
    ],
}


def has_criterion_keyword(evidence, criterion):
    evidence_lower = evidence.lower()
    keywords = CRITERION_KEYWORDS.get(criterion, [])

    return any(
        keyword in evidence_lower
        for keyword in keywords
    )


for item in criterion_evidence_table:
    evidence = item["best_evidence"]

    if evidence is None:
        continue

    keyword_match = has_criterion_keyword(
        evidence,
        item["criterion"],
    )

    print(
        f"{item['criterion']}: "
        f"keyword_match={keyword_match} | "
        f"evidence={evidence}"
    )


ROS 2 application development: keyword_match=True | evidence=Developed ROS 2 applications for an autonomous mobile robot.
ROS 2 communication: keyword_match=False | evidence=Developed ROS 2 applications for an autonomous mobile robot.
ROS 2 launch systems: keyword_match=False | evidence=Developed ROS 2 applications for an autonomous mobile robot.
Sensor integration: keyword_match=False | evidence=Combined IMU and camera data for sensor fusion.
Localization: keyword_match=True | evidence=Measured localization error and navigation success rate.
Navigation: keyword_match=True | evidence=Integrated Nav2 navigation and tested SLAM in Gazebo simulation.
SLAM: keyword_match=True | evidence=Integrated Nav2 navigation and tested SLAM in Gazebo simulation.
Visual odometry: keyword_match=False | evidence=Measured localization error and navigation success rate.
Sensor fusion: keyword_match=True | evidence=Combined IMU and camera data for sensor fusion.
Nav2: keyword_match=True | evidence=Integrate

In [855]:

# Add wording that describes sensor integration without using
# the exact phrase "sensor integration".
CRITERION_KEYWORDS["Sensor integration"] = [
    "sensor integration",
    "integrated imu",
    "integrated camera",
    "integrated sensors",
    "combined imu and camera",
    "combined sensor data",
    "integrated imu and camera",
]


def classify_criterion_evidence(
    evidence,
    criterion,
    similarity,
    minimum_similarity=0.40,
):
    if evidence is None:
        return "not_established"

    keyword_match = has_criterion_keyword(
        evidence,
        criterion,
    )

    if similarity < minimum_similarity:
        return "weak_semantic_match"

    if keyword_match:
        return "candidate_evidence_for_review"

    return "semantic_match_only"


for item in criterion_evidence_table:
    evidence = item["best_evidence"]

    if evidence is None:
        status = "not_established"
    else:
        status = classify_criterion_evidence(
            evidence=evidence,
            criterion=item["criterion"],
            similarity=item["similarity"],
        )

    print(f"{item['criterion']}: {status}")


Python development for robotics applications: not_established
C++ development for robotics applications: not_established
ROS 2 application development: candidate_evidence_for_review
ROS 2 communication: semantic_match_only
ROS 2 lifecycle management: not_established
ROS 2 launch systems: semantic_match_only
ROS 2 debugging: not_established
Robot kinematics: not_established
Robot dynamics: not_established
Feedback control: not_established
Trajectory generation: not_established
Sensor integration: candidate_evidence_for_review
Coordinate-frame management using TF2: not_established
Localization: candidate_evidence_for_review
Mapping: not_established
Navigation: candidate_evidence_for_review
Manipulation: not_established
Software testing: not_established
Performance profiling: not_established
Git workflows: not_established
Hardware/software integration: not_established
Sensor failure diagnosis: not_established
Algorithm failure diagnosis: not_established
Middleware failure diagnosis: not_e

In [856]:

def build_complete_evidence_matrix(
    cv_text,
    criteria,
    minimum_similarity=0.40,
):
    sentences = extract_cv_evidence_sentences(cv_text)

    criterion_texts = [
        item["criterion"] for item in criteria
    ]

    criterion_embeddings = embedding_model.encode(
        criterion_texts,
        batch_size=16,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)

    sentence_embeddings = embedding_model.encode(
        sentences,
        batch_size=16,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)

    similarity_matrix = sentence_embeddings @ criterion_embeddings.T
    matrix = []

    for criterion_idx, criterion_item in enumerate(criteria):
        candidates = []

        for sentence_idx, sentence in enumerate(sentences):
            similarity = float(
                similarity_matrix[sentence_idx, criterion_idx]
            )

            if similarity < minimum_similarity:
                continue

            if classify_evidence_statement(sentence) == "not_mentioned":
                continue

            candidates.append({
                "evidence": sentence,
                "similarity": similarity,
                "keyword_match": has_criterion_keyword(
                    sentence,
                    criterion_item["criterion"],
                ),
            })

        candidates.sort(
            key=lambda item: item["similarity"],
            reverse=True,
        )

        best = candidates[0] if candidates else None

        if best is None:
            status = "not_established"
        elif best["keyword_match"]:
            status = "candidate_evidence_for_review"
        else:
            status = "semantic_match_only"

        matrix.append({
            "criterion": criterion_item["criterion"],
            "source_section": criterion_item["source_section"],
            "best_evidence": best["evidence"] if best else None,
            "similarity": best["similarity"] if best else None,
            "keyword_match": best["keyword_match"] if best else False,
            "status": status,
        })

    return matrix


complete_evidence_matrix = build_complete_evidence_matrix(
    cv_text=sample_cv,
    criteria=atomic_criteria,
)

for item in complete_evidence_matrix:
    if item["best_evidence"] is not None:
        similarity = item["similarity"]

        print(
            f"{item['criterion']} | "
            f"{item['status']} | "
            f"similarity={similarity:.4f}"
        )
        print(f"  Evidence: {item['best_evidence']}")


Python development for robotics applications | semantic_match_only | similarity=0.4272
  Evidence: Developed ROS 2 applications for an autonomous mobile robot.
C++ development for robotics applications | semantic_match_only | similarity=0.4179
  Evidence: Developed ROS 2 applications for an autonomous mobile robot.
ROS 2 application development | candidate_evidence_for_review | similarity=0.6952
  Evidence: Developed ROS 2 applications for an autonomous mobile robot.
ROS 2 communication | semantic_match_only | similarity=0.6290
  Evidence: Developed ROS 2 applications for an autonomous mobile robot.
ROS 2 lifecycle management | semantic_match_only | similarity=0.6187
  Evidence: Developed ROS 2 applications for an autonomous mobile robot.
ROS 2 launch systems | semantic_match_only | similarity=0.6393
  Evidence: Developed ROS 2 applications for an autonomous mobile robot.
ROS 2 debugging | semantic_match_only | similarity=0.5230
  Evidence: Developed ROS 2 applications for an autonomou

In [857]:

def prepare_scoring_eligibility(evidence_matrix):
    eligible = []

    for item in evidence_matrix:
        if item["status"] == "candidate_evidence_for_review":
            decision = "review_candidate_evidence"
        elif item["status"] == "semantic_match_only":
            decision = "exclude_from_positive_evidence"
        else:
            decision = "no_evidence_found"

        eligible.append({
            **item,
            "scoring_decision": decision,
        })

    return eligible


scoring_eligibility = prepare_scoring_eligibility(
    complete_evidence_matrix
)

for item in scoring_eligibility:
    if item["scoring_decision"] != "no_evidence_found":
        print(
            f"{item['criterion']} | "
            f"{item['scoring_decision']}"
        )
        print(f"  Evidence: {item['best_evidence']}")

print("\n--- Summary ---")
print(
    "Candidate evidence for review:",
    sum(
        item["scoring_decision"] == "review_candidate_evidence"
        for item in scoring_eligibility
    ),
)
print(
    "Excluded semantic-only matches:",
    sum(
        item["scoring_decision"] == "exclude_from_positive_evidence"
        for item in scoring_eligibility
    ),
)
print(
    "No evidence found:",
    sum(
        item["scoring_decision"] == "no_evidence_found"
        for item in scoring_eligibility
    ),
)


Python development for robotics applications | exclude_from_positive_evidence
  Evidence: Developed ROS 2 applications for an autonomous mobile robot.
C++ development for robotics applications | exclude_from_positive_evidence
  Evidence: Developed ROS 2 applications for an autonomous mobile robot.
ROS 2 application development | review_candidate_evidence
  Evidence: Developed ROS 2 applications for an autonomous mobile robot.
ROS 2 communication | exclude_from_positive_evidence
  Evidence: Developed ROS 2 applications for an autonomous mobile robot.
ROS 2 lifecycle management | exclude_from_positive_evidence
  Evidence: Developed ROS 2 applications for an autonomous mobile robot.
ROS 2 launch systems | exclude_from_positive_evidence
  Evidence: Developed ROS 2 applications for an autonomous mobile robot.
ROS 2 debugging | exclude_from_positive_evidence
  Evidence: Developed ROS 2 applications for an autonomous mobile robot.
Robot kinematics | exclude_from_positive_evidence
  Evidence: 

In [858]:

def review_evidence_strength(criterion, evidence):
    if evidence is None:
        return "not_established"

    criterion_lower = criterion.lower()
    evidence_lower = evidence.lower()

    # Specific, directly relevant wording for this sample.
    direct_evidence_phrases = {
        "ROS 2 application development": [
            "developed ros 2 applications",
            "developed ros2 applications",
        ],
        "Sensor integration": [
            "combined imu and camera",
            "integrated imu and camera",
        ],
        "Localization": [
            "measured localization error",
            "measured localisation error",
        ],
        "Navigation": [
            "integrated nav2 navigation",
            "implemented navigation",
        ],
        "SLAM": [
            "implemented slam",
            "developed a slam",
            "tested slam",
        ],
        "Sensor fusion": [
            "sensor fusion",
            "fused imu",
        ],
        "Nav2": [
            "integrated nav2",
            "implemented nav2",
        ],
    }

    phrases = direct_evidence_phrases.get(
        criterion,
        [],
    )

    if any(phrase in evidence_lower for phrase in phrases):
        # Testing a technology is evidence of exposure, not necessarily
        # complete independent proficiency.
        if "tested slam" in evidence_lower:
            return "partial_evidence"

        return "needs_review"

    return "needs_review"


for item in scoring_eligibility:
    if item["scoring_decision"] != "review_candidate_evidence":
        continue

    strength = review_evidence_strength(
        item["criterion"],
        item["best_evidence"],
    )

    print(f"\nCriterion: {item['criterion']}")
    print(f"Evidence: {item['best_evidence']}")
    print(f"Evidence status: {strength}")



Criterion: ROS 2 application development
Evidence: Developed ROS 2 applications for an autonomous mobile robot.
Evidence status: needs_review

Criterion: Sensor integration
Evidence: Combined IMU and camera data for sensor fusion.
Evidence status: needs_review

Criterion: Localization
Evidence: Measured localization error and navigation success rate.
Evidence status: needs_review

Criterion: Navigation
Evidence: Integrated Nav2 navigation and tested SLAM in Gazebo simulation.
Evidence status: partial_evidence

Criterion: SLAM
Evidence: Integrated Nav2 navigation and tested SLAM in Gazebo simulation.
Evidence status: partial_evidence

Criterion: Sensor fusion
Evidence: Combined IMU and camera data for sensor fusion.
Evidence status: needs_review

Criterion: Nav2
Evidence: Integrated Nav2 navigation and tested SLAM in Gazebo simulation.
Evidence status: partial_evidence


In [859]:

def build_review_table(evidence_matrix, criteria):
    review_table = []

    for item, criterion_item in zip(evidence_matrix, criteria):
        evidence = item["best_evidence"]

        if evidence is None:
            evidence_status = "not_established"
        elif item["status"] == "semantic_match_only":
            evidence_status = "needs_review"
        else:
            evidence_status = review_evidence_strength(
                item["criterion"],
                evidence,
            )

        review_table.append({
            "criterion": item["criterion"],
            "importance": item["source_section"],
            "evidence": evidence,
            "similarity": item["similarity"],
            "evidence_status": evidence_status,
            "review_decision": "pending",
        })

    return review_table


review_table = build_review_table(
    evidence_matrix=complete_evidence_matrix,
    criteria=atomic_criteria,
)

print("Total criteria:", len(review_table))
print("\nCriteria with candidate evidence:")

for item in review_table:
    if item["evidence"] is not None:
        similarity = (
            f"{item['similarity']:.4f}"
            if item["similarity"] is not None
            else "N/A"
        )

        print(f"\nCriterion: {item['criterion']}")
        print(f"Importance: {item['importance']}")
        print(f"Evidence status: {item['evidence_status']}")
        print(f"Similarity (retrieval only): {similarity}")
        print(f"Evidence: {item['evidence']}")
        print(f"Review decision: {item['review_decision']}")

print("\nPASS: Review table created.")


Total criteria: 43

Criteria with candidate evidence:

Criterion: Python development for robotics applications
Importance: required_skills
Evidence status: needs_review
Similarity (retrieval only): 0.4272
Evidence: Developed ROS 2 applications for an autonomous mobile robot.
Review decision: pending

Criterion: C++ development for robotics applications
Importance: required_skills
Evidence status: needs_review
Similarity (retrieval only): 0.4179
Evidence: Developed ROS 2 applications for an autonomous mobile robot.
Review decision: pending

Criterion: ROS 2 application development
Importance: required_skills
Evidence status: needs_review
Similarity (retrieval only): 0.6952
Evidence: Developed ROS 2 applications for an autonomous mobile robot.
Review decision: pending

Criterion: ROS 2 communication
Importance: required_skills
Evidence status: needs_review
Similarity (retrieval only): 0.6290
Evidence: Developed ROS 2 applications for an autonomous mobile robot.
Review decision: pending



In [860]:

VALID_REVIEW_DECISIONS = {
    "supported",
    "partial",
    "not_supported",
    "pending",
}


def set_review_decision(review_table, criterion, decision):
    if decision not in VALID_REVIEW_DECISIONS:
        raise ValueError(
            f"Invalid decision: {decision}. "
            f"Choose from {sorted(VALID_REVIEW_DECISIONS)}"
        )

    for item in review_table:
        if item["criterion"] == criterion:
            item["review_decision"] = decision
            return

    raise ValueError(f"Criterion not found: {criterion}")


# Example: keep this evidence pending until it is reviewed.
set_review_decision(
    review_table,
    criterion="ROS 2 application development",
    decision="pending",
)

# Verify the review decision.
for item in review_table:
    if item["criterion"] == "ROS 2 application development":
        print("Criterion:", item["criterion"])
        print("Evidence:", item["evidence"])
        print("Review decision:", item["review_decision"])
        break

print("\nValid decisions:", sorted(VALID_REVIEW_DECISIONS))


Criterion: ROS 2 application development
Evidence: Developed ROS 2 applications for an autonomous mobile robot.
Review decision: pending

Valid decisions: ['not_supported', 'partial', 'pending', 'supported']


In [861]:

def calculate_evidence_coverage(review_table):
    summary = {}

    for section in ["required_skills", "preferred_skills"]:
        items = [
            item for item in review_table
            if item["importance"] == section
        ]

        summary[section] = {
            "total": len(items),
            "with_cv_evidence": sum(
                item["evidence"] is not None
                for item in items
            ),
            "supported": sum(
                item["review_decision"] == "supported"
                for item in items
            ),
            "partial": sum(
                item["review_decision"] == "partial"
                for item in items
            ),
            "not_supported": sum(
                item["review_decision"] == "not_supported"
                for item in items
            ),
            "pending": sum(
                item["review_decision"] == "pending"
                for item in items
            ),
            "no_evidence": sum(
                item["evidence"] is None
                for item in items
            ),
        }

    return summary


coverage_summary = calculate_evidence_coverage(review_table)

for section, stats in coverage_summary.items():
    print(f"\n{section.upper()}")
    for key, value in stats.items():
        print(f"  {key}: {value}")

print("\nPASS: Evidence coverage summary calculated.")



REQUIRED_SKILLS
  total: 25
  with_cv_evidence: 12
  supported: 0
  partial: 0
  not_supported: 0
  pending: 25
  no_evidence: 13

PREFERRED_SKILLS
  total: 18
  with_cv_evidence: 4
  supported: 0
  partial: 0
  not_supported: 0
  pending: 18
  no_evidence: 14

PASS: Evidence coverage summary calculated.


In [862]:

def calculate_evidence_coverage(review_table):
    summary = {}

    for section in ["required_skills", "preferred_skills"]:
        items = [
            item for item in review_table
            if item["importance"] == section
        ]

        summary[section] = {
            "total": len(items),
            "with_cv_evidence": sum(
                item["evidence"] is not None
                for item in items
            ),
            "no_evidence_found": sum(
                item["evidence"] is None
                for item in items
            ),
            "pending_with_evidence": sum(
                item["review_decision"] == "pending"
                and item["evidence"] is not None
                for item in items
            ),
            "pending_no_evidence": sum(
                item["review_decision"] == "pending"
                and item["evidence"] is None
                for item in items
            ),
            "supported": sum(
                item["review_decision"] == "supported"
                for item in items
            ),
            "partial": sum(
                item["review_decision"] == "partial"
                for item in items
            ),
            "not_supported": sum(
                item["review_decision"] == "not_supported"
                for item in items
            ),
        }

    return summary


coverage_summary = calculate_evidence_coverage(review_table)

for section, stats in coverage_summary.items():
    print(f"\n{section.upper()}")
    for key, value in stats.items():
        print(f"  {key}: {value}")

print("\nPASS: Evidence and review status are separated.")



REQUIRED_SKILLS
  total: 25
  with_cv_evidence: 12
  no_evidence_found: 13
  pending_with_evidence: 12
  pending_no_evidence: 13
  supported: 0
  partial: 0
  not_supported: 0

PREFERRED_SKILLS
  total: 18
  with_cv_evidence: 4
  no_evidence_found: 14
  pending_with_evidence: 4
  pending_no_evidence: 14
  supported: 0
  partial: 0
  not_supported: 0

PASS: Evidence and review status are separated.


In [863]:

criteria_names = {
    item["criterion"]
    for item in atomic_criteria
}

for group_name, group in ALTERNATIVE_GROUPS.items():
    group_criteria = group["criteria"]
    missing = [
        criterion
        for criterion in group_criteria
        if criterion not in criteria_names
    ]

    print(f"\nGroup: {group_name}")
    print(f"Mode: {group['mode']}")
    print(f"Reason: {group['reason']}")
    print("Criteria:")

    for criterion in group_criteria:
        status = "FOUND" if criterion in criteria_names else "MISSING"
        print(f"  - {criterion}: {status}")

    assert not missing, (
        f"Criteria missing from dataset: {missing}"
    )

print("\nPASS: All alternative groups reference valid criteria.")



Group: programming_languages
Mode: any
Reason: The original requirement says Python and/or C++.
Criteria:
  - Python development for robotics applications: FOUND
  - C++ development for robotics applications: FOUND

Group: robot_specialization
Mode: any
Reason: The original requirement says these depend on the candidate's robotics specialization.
Criteria:
  - Localization: FOUND
  - Mapping: FOUND
  - Navigation: FOUND
  - Manipulation: FOUND

Group: state_estimation_methods
Mode: any
Reason: The original requirement lists alternative methods.
Criteria:
  - SLAM: FOUND
  - Visual odometry: FOUND
  - Sensor fusion: FOUND
  - State estimation: FOUND

PASS: All alternative groups reference valid criteria.


In [864]:

def evaluate_alternative_groups(review_table, alternative_groups):
    criteria_lookup = {
        item["criterion"]: item
        for item in review_table
    }

    group_results = []

    for group_name, group in alternative_groups.items():
        items = [
            criteria_lookup[name]
            for name in group["criteria"]
        ]

        decisions = [
            item["review_decision"]
            for item in items
        ]

        has_evidence = any(
            item["evidence"] is not None
            for item in items
        )

        if "supported" in decisions:
            status = "supported"
        elif "partial" in decisions:
            status = "partial"
        elif has_evidence:
            status = "pending_with_evidence"
        else:
            status = "pending_no_evidence"

        group_results.append({
            "group": group_name,
            "mode": group["mode"],
            "status": status,
            "criteria": group["criteria"],
            "evidence_count": sum(
                item["evidence"] is not None
                for item in items
            ),
            "total_options": len(items),
        })

    return group_results


alternative_group_results = evaluate_alternative_groups(
    review_table=review_table,
    alternative_groups=ALTERNATIVE_GROUPS,
)

for result in alternative_group_results:
    print(f"\nGroup: {result['group']}")
    print(f"Mode: {result['mode']}")
    print(f"Status: {result['status']}")
    print(
        f"Options with candidate evidence: "
        f"{result['evidence_count']}/{result['total_options']}"
    )

print("\nPASS: Alternative groups evaluated.")



Group: programming_languages
Mode: any
Status: pending_with_evidence
Options with candidate evidence: 2/2

Group: robot_specialization
Mode: any
Status: pending_with_evidence
Options with candidate evidence: 2/4

Group: state_estimation_methods
Mode: any
Status: pending_with_evidence
Options with candidate evidence: 3/4

PASS: Alternative groups evaluated.


In [865]:

REVIEW_SCORES = {
    "supported": 1.0,
    "partial": 0.5,
    "not_supported": 0.0,
    "pending": None,
}


def score_reviewed_items(items):
    reviewed_scores = [
        REVIEW_SCORES[item["review_decision"]]
        for item in items
        if REVIEW_SCORES[item["review_decision"]] is not None
    ]

    if not reviewed_scores:
        return {
            "score": None,
            "reviewed": 0,
            "total": len(items),
        }

    return {
        "score": sum(reviewed_scores) / len(reviewed_scores),
        "reviewed": len(reviewed_scores),
        "total": len(items),
    }


def score_alternative_group(group, review_table):
    lookup = {
        item["criterion"]: item
        for item in review_table
    }

    items = [
        lookup[name]
        for name in group["criteria"]
    ]

    reviewed = [
        item for item in items
        if item["review_decision"] != "pending"
    ]

    if not reviewed:
        return {
            "score": None,
            "reviewed_options": 0,
            "total_options": len(items),
        }

    if group["mode"] == "any":
        score = max(
            REVIEW_SCORES[item["review_decision"]]
            for item in reviewed
        )
    else:
        score = sum(
            REVIEW_SCORES[item["review_decision"]]
            for item in reviewed
        ) / len(reviewed)

    return {
        "score": score,
        "reviewed_options": len(reviewed),
        "total_options": len(items),
    }


print("Scoring rules:")
for decision, score in REVIEW_SCORES.items():
    print(f"  {decision}: {score}")

print("\nAlternative group scoring test:")
for name, group in ALTERNATIVE_GROUPS.items():
    result = score_alternative_group(group, review_table)
    print(
        f"{name}: score={result['score']}, "
        f"reviewed={result['reviewed_options']}/"
        f"{result['total_options']}"
    )


Scoring rules:
  supported: 1.0
  partial: 0.5
  not_supported: 0.0
  pending: None

Alternative group scoring test:
programming_languages: score=None, reviewed=0/2
robot_specialization: score=None, reviewed=0/4
state_estimation_methods: score=None, reviewed=0/4


In [866]:

def calculate_role_evaluation(
    review_table,
    alternative_groups,
    required_weight=0.70,
    preferred_weight=0.30,
):
    if required_weight < 0 or preferred_weight < 0:
        raise ValueError("Weights must be non-negative.")

    if abs(required_weight + preferred_weight - 1.0) > 1e-9:
        raise ValueError("Weights must sum to 1.0.")

    grouped_criteria = {
        criterion
        for group in alternative_groups.values()
        for criterion in group["criteria"]
    }

    results = {}

    for section, section_weight in [
        ("required_skills", required_weight),
        ("preferred_skills", preferred_weight),
    ]:
        standalone = [
            item for item in review_table
            if item["importance"] == section
            and item["criterion"] not in grouped_criteria
        ]

        standalone_result = score_reviewed_items(standalone)

        section_groups = []
        for group_name, group in alternative_groups.items():
            group_sections = {
                item["importance"]
                for item in review_table
                if item["criterion"] in group["criteria"]
            }

            if section in group_sections:
                group_result = score_alternative_group(
                    group,
                    review_table,
                )

                section_groups.append({
                    "name": group_name,
                    **group_result,
                })

        component_scores = []

        if standalone_result["score"] is not None:
            component_scores.append(standalone_result["score"])

        component_scores.extend(
            group["score"]
            for group in section_groups
            if group["score"] is not None
        )

        section_score = (
            sum(component_scores) / len(component_scores)
            if component_scores
            else None
        )

        results[section] = {
            "weight": section_weight,
            "score": section_score,
            "standalone_reviewed": standalone_result["reviewed"],
            "standalone_total": standalone_result["total"],
            "groups": section_groups,
            "scored_components": len(component_scores),
        }

    available_sections = [
        result
        for result in results.values()
        if result["score"] is not None
    ]

    # Do not redistribute the weight of an unscored section.
    overall_score = (
        sum(
            result["score"] * result["weight"]
            for result in available_sections
        )
        if available_sections
        else None
    )

    return {
        "overall_score": overall_score,
        "required": results["required_skills"],
        "preferred": results["preferred_skills"],
    }


role_evaluation = calculate_role_evaluation(
    review_table=review_table,
    alternative_groups=ALTERNATIVE_GROUPS,
)

print("Overall score:", role_evaluation["overall_score"])

for section_name in ["required", "preferred"]:
    section = role_evaluation[section_name]

    print(f"\n{section_name.upper()}")
    print("Score:", section["score"])
    print(
        "Standalone criteria reviewed:",
        f"{section['standalone_reviewed']}/"
        f"{section['standalone_total']}",
    )
    print("Scored components:", section["scored_components"])

    for group in section["groups"]:
        print(
            f"  Group {group['name']}: "
            f"score={group['score']}, "
            f"reviewed={group['reviewed_options']}/"
            f"{group['total_options']}"
        )

print("\nPASS: Role evaluation engine executed.")


Overall score: None

REQUIRED
Score: None
Standalone criteria reviewed: 0/19
Scored components: 0
  Group programming_languages: score=None, reviewed=0/2
  Group robot_specialization: score=None, reviewed=0/4

PREFERRED
Score: None
Standalone criteria reviewed: 0/14
Scored components: 0
  Group state_estimation_methods: score=None, reviewed=0/4

PASS: Role evaluation engine executed.


In [867]:

# Work on a separate copy so review_table remains unchanged.
synthetic_review_table = [
    dict(item) for item in review_table
]

synthetic_decisions = {
    # Alternative group: Python OR C++
    "Python development for robotics applications": "supported",
    "C++ development for robotics applications": "not_supported",

    # Alternative group: choose one robotics specialization
    "Localization": "partial",
    "Navigation": "not_supported",

    # Preferred alternative group: one supported method is sufficient
    "Sensor fusion": "supported",
    "SLAM": "partial",

    # One standalone required criterion
    "ROS 2 application development": "supported",
}

for criterion, decision in synthetic_decisions.items():
    set_review_decision(
        synthetic_review_table,
        criterion,
        decision,
    )

test_evaluation = calculate_role_evaluation(
    review_table=synthetic_review_table,
    alternative_groups=ALTERNATIVE_GROUPS,
)

for group_name, expected_score in [
    ("programming_languages", 1.0),
    ("robot_specialization", 0.5),
]:
    result = next(
        group
        for group in test_evaluation["required"]["groups"]
        if group["name"] == group_name
    )
    assert abs(result["score"] - expected_score) < 1e-9
    print(f"PASS: {group_name} score = {result['score']:.2f}")

state_group = test_evaluation["preferred"]["groups"][0]
assert state_group["name"] == "state_estimation_methods"
assert abs(state_group["score"] - 1.0) < 1e-9
print(f"PASS: state_estimation_methods score = {state_group['score']:.2f}")

assert abs(test_evaluation["required"]["score"] - (5 / 6)) < 1e-9
assert abs(test_evaluation["preferred"]["score"] - 1.0) < 1e-9
assert abs(test_evaluation["overall_score"] - (53 / 60)) < 1e-9

print(f"\nSynthetic overall score: {test_evaluation['overall_score']:.4f}")
print("PASS: All synthetic scoring tests passed.")

# Confirm the real review table was not changed.
assert all(
    item["review_decision"] == "pending"
    for item in review_table
)
print("PASS: Original review decisions remain unchanged.")


PASS: programming_languages score = 1.00
PASS: robot_specialization score = 0.50
PASS: state_estimation_methods score = 1.00

Synthetic overall score: 0.8833
PASS: All synthetic scoring tests passed.
PASS: Original review decisions remain unchanged.


In [868]:

def summarize_review_coverage(review_table, alternative_groups):
    grouped_criteria = {
        criterion
        for group in alternative_groups.values()
        for criterion in group["criteria"]
    }

    report = {}

    for section in ["required_skills", "preferred_skills"]:
        section_items = [
            item for item in review_table
            if item["importance"] == section
        ]

        reviewed_criteria = [
            item for item in section_items
            if item["review_decision"] != "pending"
        ]

        standalone = [
            item for item in section_items
            if item["criterion"] not in grouped_criteria
        ]

        reviewed_standalone = [
            item for item in standalone
            if item["review_decision"] != "pending"
        ]

        section_groups = [
            group for group in alternative_groups.values()
            if any(
                item["criterion"] in group["criteria"]
                and item["importance"] == section
                for item in section_items
            )
        ]

        reviewed_groups = 0
        for group in section_groups:
            if any(
                item["criterion"] in group["criteria"]
                and item["importance"] == section
                and item["review_decision"] != "pending"
                for item in section_items
            ):
                reviewed_groups += 1

        total_components = len(standalone) + len(section_groups)
        reviewed_components = (
            len(reviewed_standalone) + reviewed_groups
        )

        report[section] = {
            "criteria_reviewed": len(reviewed_criteria),
            "criteria_total": len(section_items),
            "criterion_coverage": (
                len(reviewed_criteria) / len(section_items)
                if section_items else 0.0
            ),
            "components_reviewed": reviewed_components,
            "components_total": total_components,
            "component_coverage": (
                reviewed_components / total_components
                if total_components else 0.0
            ),
        }

    return report


coverage_report = summarize_review_coverage(
    synthetic_review_table,
    ALTERNATIVE_GROUPS,
)

for section_name, data in coverage_report.items():
    print(f"\n{section_name.upper()}")
    print(
        f"Criteria reviewed: {data['criteria_reviewed']}/"
        f"{data['criteria_total']} "
        f"({data['criterion_coverage']:.1%})"
    )
    print(
        f"Scoring components reviewed: "
        f"{data['components_reviewed']}/"
        f"{data['components_total']} "
        f"({data['component_coverage']:.1%})"
    )

print("\nPASS: Coverage report generated.")



REQUIRED_SKILLS
Criteria reviewed: 5/25 (20.0%)
Scoring components reviewed: 3/21 (14.3%)

PREFERRED_SKILLS
Criteria reviewed: 2/18 (11.1%)
Scoring components reviewed: 1/15 (6.7%)

PASS: Coverage report generated.


In [869]:
def score_alternative_group(group, review_table):
    lookup = {
        item["criterion"]: item
        for item in review_table
    }

    items = [
        lookup[name]
        for name in group["criteria"]
    ]

    decisions = [
        item["review_decision"]
        for item in items
    ]

    # For an "any" group, one supported option is sufficient.
    if group["mode"] == "any":
        if "supported" in decisions:
            score = 1.0
        elif "pending" in decisions:
            # An unreviewed alternative could still satisfy the requirement.
            score = None
        elif "partial" in decisions:
            score = 0.5
        else:
            # Every alternative was reviewed and none was supported.
            score = 0.0

        return {
            "score": score,
            "reviewed_options": sum(
                decision != "pending"
                for decision in decisions
            ),
            "total_options": len(items),
        }

    raise NotImplementedError(
        f"Scoring mode not implemented: {group['mode']}"
    )


# Test 1: Python failed, but C++ is still unreviewed.
test_group = ALTERNATIVE_GROUPS["programming_languages"]

test_table = [
    dict(item) for item in review_table
]

set_review_decision(
    test_table,
    "Python development for robotics applications",
    "not_supported",
)

result = score_alternative_group(test_group, test_table)
assert result["score"] is None
print("PASS: One failed option + one pending option = pending")

# Test 2: One supported option is enough.
set_review_decision(
    test_table,
    "C++ development for robotics applications",
    "supported",
)

result = score_alternative_group(test_group, test_table)
assert result["score"] == 1.0
print("PASS: One supported alternative satisfies the group")

# Test 3: All alternatives reviewed and none supported.
set_review_decision(
    test_table,
    "C++ development for robotics applications",
    "not_supported",
)

result = score_alternative_group(test_group, test_table)
assert result["score"] == 0.0
print("PASS: All alternatives rejected = zero")

# Confirm the original review table remains unchanged.
assert all(
    item["review_decision"] == "pending"
    for item in review_table
)
print("PASS: Original review table remains unchanged")

PASS: One failed option + one pending option = pending
PASS: One supported alternative satisfies the group
PASS: All alternatives rejected = zero
PASS: Original review table remains unchanged


In [870]:

def calculate_role_evaluation(
    review_table,
    alternative_groups,
    required_weight=0.70,
    preferred_weight=0.30,
):
    if abs(required_weight + preferred_weight - 1.0) > 1e-9:
        raise ValueError("Section weights must sum to 1.0.")

    grouped_criteria = {
        criterion
        for group in alternative_groups.values()
        for criterion in group["criteria"]
    }

    def evaluate_section(section_name):
        section_items = [
            item for item in review_table
            if item["importance"] == section_name
        ]

        standalone = [
            item for item in section_items
            if item["criterion"] not in grouped_criteria
        ]

        standalone_result = score_reviewed_items(standalone)

        group_results = []
        for group_name, group in alternative_groups.items():
            group_items = [
                item for item in section_items
                if item["criterion"] in group["criteria"]
            ]

            if not group_items:
                continue

            result = score_alternative_group(group, review_table)
            group_results.append({
                "name": group_name,
                **result,
            })

        components = []

        if standalone_result["score"] is not None:
            components.append(standalone_result["score"])

        components.extend(
            result["score"]
            for result in group_results
            if result["score"] is not None
        )

        section_score = (
            sum(components) / len(components)
            if components else None
        )

        reviewed_criteria = sum(
            item["review_decision"] != "pending"
            for item in section_items
        )

        return {
            "score": section_score,
            "criteria_reviewed": reviewed_criteria,
            "criteria_total": len(section_items),
            "criterion_coverage": (
                reviewed_criteria / len(section_items)
                if section_items else 0.0
            ),
            "standalone": standalone_result,
            "groups": group_results,
            "scoring_components_reviewed": len(components),
            "scoring_components_total": (
                (1 if standalone else 0) + len(group_results)
            ),
        }

    required = evaluate_section("required_skills")
    preferred = evaluate_section("preferred_skills")

    # Do not treat an entirely unreviewed section as a zero.
    # Only calculate the overall score when both sections have a score.
    if required["score"] is not None and preferred["score"] is not None:
        overall_score = (
            required_weight * required["score"]
            + preferred_weight * preferred["score"]
        )
    else:
        overall_score = None

    return {
        "overall_score": overall_score,
        "status": (
            "scored"
            if overall_score is not None
            else "incomplete_review"
        ),
        "required": required,
        "preferred": preferred,
        "weights": {
            "required": required_weight,
            "preferred": preferred_weight,
        },
    }


# Test with the synthetic decisions from the earlier test.
updated_test = calculate_role_evaluation(
    review_table=synthetic_review_table,
    alternative_groups=ALTERNATIVE_GROUPS,
)

print("Status:", updated_test["status"])
print("Required coverage:",
      f"{updated_test['required']['criterion_coverage']:.1%}")
print("Preferred coverage:",
      f"{updated_test['preferred']['criterion_coverage']:.1%}")
print("Synthetic overall score:",
      updated_test["overall_score"])

assert updated_test["status"] == "scored"
assert updated_test["overall_score"] is not None

# The real table still has no completed reviews.
real_test = calculate_role_evaluation(
    review_table=review_table,
    alternative_groups=ALTERNATIVE_GROUPS,
)

assert real_test["status"] == "incomplete_review"
assert real_test["overall_score"] is None

print("\nPASS: Synthetic scoring works.")
print("PASS: Unreviewed real table receives no overall score.")


Status: scored
Required coverage: 20.0%
Preferred coverage: 11.1%
Synthetic overall score: 1.0

PASS: Synthetic scoring works.
PASS: Unreviewed real table receives no overall score.


In [871]:

# Save the current function's logic by redefining it with stricter
# completeness requirements.

def calculate_role_evaluation(
    review_table,
    alternative_groups,
    required_weight=0.70,
    preferred_weight=0.30,
):
    if abs(required_weight + preferred_weight - 1.0) > 1e-9:
        raise ValueError("Section weights must sum to 1.0.")

    grouped_criteria = {
        criterion
        for group in alternative_groups.values()
        for criterion in group["criteria"]
    }

    def evaluate_section(section_name):
        items = [
            item for item in review_table
            if item["importance"] == section_name
        ]

        standalone = [
            item for item in items
            if item["criterion"] not in grouped_criteria
        ]

        standalone_result = score_reviewed_items(standalone)

        group_results = []
        for group_name, group in alternative_groups.items():
            if not any(
                item["criterion"] in group["criteria"]
                for item in items
            ):
                continue

            result = score_alternative_group(group, review_table)
            group_results.append({
                "name": group_name,
                **result,
            })

        standalone_complete = all(
            item["review_decision"] != "pending"
            for item in standalone
        )

        groups_complete = all(
            group["score"] is not None
            for group in group_results
        )

        section_complete = standalone_complete and groups_complete

        # Only publish a section score when its review is complete.
        section_score = None
        if section_complete:
            component_scores = []

            if standalone:
                component_scores.append(standalone_result["score"])

            component_scores.extend(
                group["score"] for group in group_results
            )

            if component_scores:
                section_score = (
                    sum(component_scores) / len(component_scores)
                )

        reviewed = sum(
            item["review_decision"] != "pending"
            for item in items
        )

        return {
            "score": section_score,
            "status": "complete" if section_complete else "incomplete",
            "criteria_reviewed": reviewed,
            "criteria_total": len(items),
            "criterion_coverage": (
                reviewed / len(items) if items else 0.0
            ),
            "standalone": standalone_result,
            "groups": group_results,
        }

    required = evaluate_section("required_skills")
    preferred = evaluate_section("preferred_skills")

    complete = (
        required["status"] == "complete"
        and preferred["status"] == "complete"
    )

    overall_score = None
    if complete:
        overall_score = (
            required_weight * required["score"]
            + preferred_weight * preferred["score"]
        )

    return {
        "overall_score": overall_score,
        "status": "scored" if complete else "incomplete_review",
        "required": required,
        "preferred": preferred,
        "weights": {
            "required": required_weight,
            "preferred": preferred_weight,
        },
    }


# Check that low-coverage synthetic data is not given a final score.
test_result = calculate_role_evaluation(
    synthetic_review_table,
    ALTERNATIVE_GROUPS,
)

print("Synthetic status:", test_result["status"])
print("Synthetic overall score:", test_result["overall_score"])

assert test_result["status"] == "incomplete_review"
assert test_result["overall_score"] is None

# Check that the actual, entirely unreviewed table is also incomplete.
real_result = calculate_role_evaluation(
    review_table,
    ALTERNATIVE_GROUPS,
)

assert real_result["status"] == "incomplete_review"
assert real_result["overall_score"] is None

print("\nPASS: Incomplete reviews cannot produce a final score.")
print("PASS: Original review decisions remain unchanged.")


Synthetic status: incomplete_review
Synthetic overall score: None

PASS: Incomplete reviews cannot produce a final score.
PASS: Original review decisions remain unchanged.


In [872]:

complete_test_table = [
    dict(item) for item in review_table
]

# Mark every criterion as reviewed in the synthetic test.
# These decisions are invented solely to test the scoring logic.
for item in complete_test_table:
    item["review_decision"] = "supported"

complete_result = calculate_role_evaluation(
    review_table=complete_test_table,
    alternative_groups=ALTERNATIVE_GROUPS,
)

print("Status:", complete_result["status"])
print("Required score:", complete_result["required"]["score"])
print("Preferred score:", complete_result["preferred"]["score"])
print("Overall score:", complete_result["overall_score"])

assert complete_result["status"] == "scored"
assert complete_result["required"]["score"] == 1.0
assert complete_result["preferred"]["score"] == 1.0
assert complete_result["overall_score"] == 1.0

# Confirm no actual review decisions were modified.
assert all(
    item["review_decision"] == "pending"
    for item in review_table
)

print("\nPASS: Complete synthetic review produces a score.")
print("PASS: Original review table remains unchanged.")


Status: scored
Required score: 1.0
Preferred score: 1.0
Overall score: 1.0

PASS: Complete synthetic review produces a score.
PASS: Original review table remains unchanged.


In [873]:

def build_evaluation_report(
    review_table,
    evaluation_result,
    role,
    level,
):
    decision_labels = {
        "supported": "Supported",
        "partial": "Partially supported",
        "not_supported": "Not supported",
        "pending": "Pending review",
    }

    report = {
        "role": role,
        "level": level,
        "status": evaluation_result["status"],
        "overall_score": evaluation_result["overall_score"],
        "sections": {},
        "findings": {
            "supported": [],
            "partial": [],
            "not_supported": [],
            "pending": [],
        },
        "limitations": [
            "CV evidence must be reviewed before a skill is confirmed.",
            "Semantic similarity is a retrieval aid, not proof of competence.",
            "A missing CV statement means the skill is not established by this CV; it does not prove the candidate lacks the skill.",
        ],
    }

    # Map report section names to the keys returned by the scorer.
    section_mapping = {
        "required_skills": "required",
        "preferred_skills": "preferred",
    }

    for report_name, result_key in section_mapping.items():
        section_result = evaluation_result[result_key]

        report["sections"][report_name] = {
            "status": section_result["status"],
            "score": section_result["score"],
            "criteria_reviewed": section_result["criteria_reviewed"],
            "criteria_total": section_result["criteria_total"],
            "coverage": section_result["criterion_coverage"],
        }

    for item in review_table:
        decision = item["review_decision"]

        report["findings"][decision].append({
            "criterion": item["criterion"],
            "importance": item["importance"],
            "evidence": item["evidence"],
            "evidence_status": item["evidence_status"],
            "review_label": decision_labels[decision],
        })

    return report


# Rebuild the report using the existing evaluation result.
current_report = build_evaluation_report(
    review_table=review_table,
    evaluation_result=real_result,
    role="Robotics Engineer",
    level="mid_level",
)

print("Role:", current_report["role"])
print("Level:", current_report["level"])
print("Status:", current_report["status"])
print("Overall score:", current_report["overall_score"])

print("\nReview findings:")
for category, items in current_report["findings"].items():
    print(f"- {category}: {len(items)}")

print("\nSection coverage:")
for section_name, section in current_report["sections"].items():
    print(
        f"- {section_name}: "
        f"{section['criteria_reviewed']}/"
        f"{section['criteria_total']} reviewed "
        f"({section['coverage']:.1%})"
    )

assert current_report["overall_score"] is None
assert len(current_report["findings"]["pending"]) == len(review_table)

print("\nPASS: Report generated without inventing a candidate score.")

       


# Build a report from the real, still-unreviewed table.
current_report = build_evaluation_report(
    review_table=review_table,
    evaluation_result=real_result,
    role="Robotics Engineer",
    level="mid_level",
)

print("Role:", current_report["role"])
print("Level:", current_report["level"])
print("Status:", current_report["status"])
print("Overall score:", current_report["overall_score"])

print("\nReview findings:")
for category, items in current_report["findings"].items():
    print(f"- {category}: {len(items)}")

print("\nLimitations:")
for limitation in current_report["limitations"]:
    print("-", limitation)

assert current_report["overall_score"] is None
assert len(current_report["findings"]["pending"]) == len(review_table)

print("\nPASS: Report generated without inventing a candidate score.")


Role: Robotics Engineer
Level: mid_level
Status: incomplete_review
Overall score: None

Review findings:
- supported: 0
- partial: 0
- not_supported: 0
- pending: 43

Section coverage:
- required_skills: 0/25 reviewed (0.0%)
- preferred_skills: 0/18 reviewed (0.0%)

PASS: Report generated without inventing a candidate score.
Role: Robotics Engineer
Level: mid_level
Status: incomplete_review
Overall score: None

Review findings:
- supported: 0
- partial: 0
- not_supported: 0
- pending: 43

Limitations:
- CV evidence must be reviewed before a skill is confirmed.
- Semantic similarity is a retrieval aid, not proof of competence.
- A missing CV statement means the skill is not established by this CV; it does not prove the candidate lacks the skill.

PASS: Report generated without inventing a candidate score.


In [874]:

def display_evaluation_summary(report):
    print("=" * 55)
    print("CV EVALUATION REPORT")
    print("=" * 55)

    print(f"Target role: {report['role']}")
    print(f"Target level: {report['level']}")
    print(f"Assessment status: {report['status']}")

    score = report["overall_score"]
    if score is None:
        print("Overall score: Not available — review incomplete")
    else:
        print(f"Overall score: {score:.1%}")

    print("\nSECTION COVERAGE")
    for name, section in report["sections"].items():
        readable_name = name.replace("_", " ").title()
        print(
            f"{readable_name}: "
            f"{section['criteria_reviewed']}/"
            f"{section['criteria_total']} criteria reviewed "
            f"({section['coverage']:.1%})"
        )

    print("\nREVIEW STATUS")
    for status, items in report["findings"].items():
        print(f"{status.replace('_', ' ').title()}: {len(items)}")

    print("\nIMPORTANT LIMITATIONS")
    for limitation in report["limitations"]:
        print(f"- {limitation}")

    print("=" * 55)


display_evaluation_summary(current_report)


CV EVALUATION REPORT
Target role: Robotics Engineer
Target level: mid_level
Assessment status: incomplete_review
Overall score: Not available — review incomplete

SECTION COVERAGE
Required Skills: 0/25 criteria reviewed (0.0%)
Preferred Skills: 0/18 criteria reviewed (0.0%)

REVIEW STATUS
Supported: 0
Partial: 0
Not Supported: 0
Pending: 43

IMPORTANT LIMITATIONS
- CV evidence must be reviewed before a skill is confirmed.
- Semantic similarity is a retrieval aid, not proof of competence.
- A missing CV statement means the skill is not established by this CV; it does not prove the candidate lacks the skill.


In [875]:

def build_review_queue(review_table):
    queue = []

    for item in review_table:
        evidence = item.get("evidence")
        similarity = item.get("similarity")

        queue.append({
            "criterion": item["criterion"],
            "importance": item["importance"],
            "cv_evidence": evidence,
            "evidence_status": item["evidence_status"],
            "similarity": similarity,
            "decision": item["review_decision"],
        })

    # Put criteria with retrieved evidence first.
    # Similarity is only a sorting aid, not proof of skill.
    queue.sort(
        key=lambda item: (
            item["cv_evidence"] is not None,
            item["similarity"] if item["similarity"] is not None else -1,
        ),
        reverse=True,
    )

    return queue


review_queue = build_review_queue(review_table)

print("Total review items:", len(review_queue))
print("\nFirst 5 items:")

for i, item in enumerate(review_queue[:5], start=1):
    print(f"\n{i}. {item['criterion']}")
    print("   Importance:", item["importance"])
    print("   Evidence status:", item["evidence_status"])
    print("   Similarity:", item["similarity"])
    print("   CV evidence:", item["cv_evidence"])
    print("   Decision:", item["decision"])

assert len(review_queue) == len(review_table)
assert all(item["decision"] == "pending" for item in review_queue)

print("\nPASS: Review queue built without changing decisions.")


Total review items: 43

First 5 items:

1. Sensor fusion
   Importance: preferred_skills
   Evidence status: needs_review
   Similarity: 0.729792058467865
   CV evidence: Combined IMU and camera data for sensor fusion.
   Decision: pending

2. ROS 2 application development
   Importance: required_skills
   Evidence status: needs_review
   Similarity: 0.6951960325241089
   CV evidence: Developed ROS 2 applications for an autonomous mobile robot.
   Decision: pending

3. ROS 2 launch systems
   Importance: required_skills
   Evidence status: needs_review
   Similarity: 0.6392831802368164
   CV evidence: Developed ROS 2 applications for an autonomous mobile robot.
   Decision: pending

4. ROS 2 communication
   Importance: required_skills
   Evidence status: needs_review
   Similarity: 0.6290155053138733
   CV evidence: Developed ROS 2 applications for an autonomous mobile robot.
   Decision: pending

5. ROS 2 lifecycle management
   Importance: required_skills
   Evidence status: needs_r

In [876]:

def assess_evidence_relevance(evidence, criterion):
    if not evidence:
        return {
            "relevance": "no_evidence",
            "reason": "No matching CV statement was found.",
        }

    if has_criterion_keyword(evidence, criterion):
        return {
            "relevance": "keyword_match",
            "reason": (
                "The CV statement contains a criterion-specific "
                "keyword or phrase. A reviewer must still verify context."
            ),
        }

    return {
        "relevance": "semantic_only",
        "reason": (
            "The match is based on semantic similarity without a "
            "recognized criterion-specific keyword. Do not treat it "
            "as evidence that the skill is demonstrated."
        ),
    }


def build_review_queue(review_table):
    queue = []

    for item in review_table:
        evidence = item.get("evidence")
        criterion = item["criterion"]

        relevance = assess_evidence_relevance(evidence, criterion)

        queue.append({
            "criterion": criterion,
            "importance": item["importance"],
            "cv_evidence": evidence,
            "similarity": item.get("similarity"),
            "evidence_status": item["evidence_status"],
            "relevance": relevance["relevance"],
            "relevance_reason": relevance["reason"],
            "decision": item["review_decision"],
        })

    relevance_order = {
        "keyword_match": 0,
        "semantic_only": 1,
        "no_evidence": 2,
    }

    queue.sort(
        key=lambda item: (
            relevance_order[item["relevance"]],
            -(item["similarity"] or 0.0),
        )
    )

    return queue


review_queue = build_review_queue(review_table)

for item in review_queue[:8]:
    print(f"\nCriterion: {item['criterion']}")
    print("Relevance:", item["relevance"])
    print("Evidence:", item["cv_evidence"])
    print("Reason:", item["relevance_reason"])

from collections import Counter

relevance_counts = Counter(
    item["relevance"] for item in review_queue
)

print("\nRelevance summary:")
for category in ["keyword_match", "semantic_only", "no_evidence"]:
    print(f"- {category}: {relevance_counts[category]}")

assert len(review_queue) == 43
assert all(item["decision"] == "pending" for item in review_queue)

print("\nPASS: Relevance labels added; review decisions unchanged.")



Criterion: Sensor fusion
Relevance: keyword_match
Evidence: Combined IMU and camera data for sensor fusion.
Reason: The CV statement contains a criterion-specific keyword or phrase. A reviewer must still verify context.

Criterion: ROS 2 application development
Relevance: keyword_match
Evidence: Developed ROS 2 applications for an autonomous mobile robot.
Reason: The CV statement contains a criterion-specific keyword or phrase. A reviewer must still verify context.

Criterion: Localization
Relevance: keyword_match
Evidence: Measured localization error and navigation success rate.
Reason: The CV statement contains a criterion-specific keyword or phrase. A reviewer must still verify context.

Criterion: Navigation
Relevance: keyword_match
Evidence: Integrated Nav2 navigation and tested SLAM in Gazebo simulation.
Reason: The CV statement contains a criterion-specific keyword or phrase. A reviewer must still verify context.

Criterion: Nav2
Relevance: keyword_match
Evidence: Integrated Na

In [877]:

%pip install -q gradio pymupdf

import gradio as gr
import fitz

print("Gradio:", gr.__version__)
print("PyMuPDF:", fitz.VersionBind)
print("Knowledge-base sections:", len(chunks))
print("Available roles:", ROLES)
print("Available levels:", LEVELS)

assert len(chunks) == 72
print("\nPASS: Upload-interface libraries and knowledge base are ready.")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.6 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.33.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Gradio: 6.26.0
PyMuPDF: 1.28.2
Knowledge-base sections: 72
Available roles: ['Cybersecurity Engineer', 'Robotics Engineer', 'Generative AI Engineer', 'Digital IC Design Engineer']
Available levels: ['junior', 'mid_level', 'senior']

PASS: Upload-interface libraries and knowledge base are ready.


In [878]:

%pip install -q "transformers>=4.45,<5" accelerate


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [879]:

%pip install -q streamlit pyngrok pymupdf


Note: you may need to restart the kernel to use updated packages.


In [904]:

from pathlib import Path

APP_DIR = Path("/kaggle/working/cv_evaluator")
APP_DIR.mkdir(parents=True, exist_ok=True)

app_code = r'''
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf
import streamlit as st
import torch
from sentence_transformers import SentenceTransformer

APP_DIR = Path(__file__).parent
ROLES = [
    "Cybersecurity Engineer",
    "Robotics Engineer",
    "Generative AI Engineer",
    "Digital IC Design Engineer",
]
LEVELS = ["junior", "mid_level", "senior"]
SECTION_NAMES = {
    "required_skills": "Required Skills",
    "preferred_skills": "Preferred Skills",
    "education_experience": "Education & Experience",
    "certifications": "Certifications",
    "projects": "Recommended Projects",
    "cv_evidence": "CV Evidence Guidelines",
}
WEIGHTS = {"Required Skills": 0.70, "Preferred Skills": 0.30}

st.set_page_config(
    page_title="FALCON-GPT",
    page_icon="🦅",
    layout="wide",
    initial_sidebar_state="expanded",
)

st.markdown("""
<style>
.stApp { background: #f4f7fb; }
[data-testid="stSidebar"] { background: #13213a; }
[data-testid="stSidebar"] * { color: #f4f7fb !important; }
.hero {
  padding: 30px; border-radius: 18px; color: white;
  background: linear-gradient(120deg,#14233f,#245b86);
  margin-bottom: 20px;
}
.hero h1 { color:white; margin-bottom:8px; }
.hero p { color:#e0ebf8; margin:0; }
.metric {
  background:white; padding:18px; border:1px solid #e1e8f0;
  border-radius:14px; min-height:105px;
}
.metric-label { color:#64748b; font-size:.85rem; }
.metric-value { color:#183153; font-size:1.8rem; font-weight:750; }
div.stButton > button[kind="primary"] {
  background:#2457a7; border:0; border-radius:9px; min-height:44px;
}
</style>
""", unsafe_allow_html=True)


@st.cache_data
def load_kb():
    path = APP_DIR / "knowledge_base.json"
    with open(path, encoding="utf-8") as f:
        return json.load(f)


@st.cache_resource
def load_embedding_model():
    return SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


@st.cache_resource
def load_chat_model():
    # Loaded only when the user sends the first chat message.
    from transformers import AutoTokenizer, AutoModelForCausalLM

    model_name = "Qwen/Qwen2.5-1.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map="auto",
    )
    model.eval()
    return tokenizer, model


def extract_pdf(upload):
    with pymupdf.open(stream=upload.getvalue(), filetype="pdf") as pdf:
        pages = [page.get_text("text") for page in pdf]
    text = "\n".join(pages)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    if len(text.strip()) < 80:
        raise ValueError(
            "Very little text was extracted. This may be a scanned PDF; "
            "OCR is needed before evaluation."
        )
    return text


def split_sentences(text):
    output = []
    for raw in text.splitlines():
        line = re.sub(r"^\s*(?:[-*•▪◦]|\d+[.)])\s*", "", raw).strip()
        if not line:
            continue
        # Join sentence fragments while preserving bullet/paragraph boundaries.
        pieces = re.split(r"(?<=[.!?])\s+", line)
        output.extend(p.strip() for p in pieces if len(p.strip()) > 8)
    return output


def get_criteria(text):
    lines = []
    for raw in text.splitlines():
        line = re.sub(r"^\s*(?:[-*•▪◦]|\d+[.)])\s*", "", raw).strip()
        line = re.sub(r"^#+\s*", "", line).strip(" ;")
        if len(line) > 12:
            lines.append(line)

    # If the section has explicit bullets, retain them as separate requirements.
    has_bullets = any(
        re.match(r"^\s*(?:[-*•▪◦]|\d+[.)])\s+", x)
        for x in text.splitlines() if x.strip()
    )
    if has_bullets:
        return list(dict.fromkeys(lines))

    result = []
    for line in lines:
        result.extend(
            p.strip() for p in re.split(r"(?<=[.!?])\s+", line)
            if len(p.strip()) > 12
        )
    return list(dict.fromkeys(result or lines))


STOPWORDS = set("""
a an the and or for to of in on with from by is are be as at into their
this that these those experience understanding knowledge basic strong
ability familiar familiarity including such relevant using use
""".split())


def lexical_overlap(criterion, evidence):
    words = lambda s: {
        w for w in re.findall(r"[a-zA-Z0-9+#.]+", s.lower())
        if len(w) > 1 and w not in STOPWORDS
    }
    required = words(criterion)
    present = words(evidence)
    if not required:
        return 0.0
    return len(required & present) / len(required)


def select_role_chunks(kb, role, level):
    return [
        item for item in kb
        if item.get("role") == role and item.get("level") == level
    ]


def evaluate_cv(cv_text, role, level, kb, embedder):
    sentences = split_sentences(cv_text)
    if not sentences:
        raise ValueError("No usable CV statements were extracted.")

    selected = select_role_chunks(kb, role, level)
    if not selected:
        raise ValueError(f"No knowledge-base data found for {role}, {level}.")

    requirements = []
    for item in selected:
        section = item.get("section", "")
        if section not in ("required_skills", "preferred_skills"):
            continue
        for criterion in get_criteria(item.get("text", "")):
            requirements.append({
                "Criterion": criterion,
                "Section": SECTION_NAMES.get(section, section),
            })

    if not requirements:
        raise ValueError("No requirements could be extracted from the knowledge base.")

    cv_vectors = embedder.encode(
        sentences, convert_to_numpy=True, normalize_embeddings=True
    ).astype(np.float32)
    requirement_vectors = embedder.encode(
        [r["Criterion"] for r in requirements],
        convert_to_numpy=True, normalize_embeddings=True,
    ).astype(np.float32)

    sims = requirement_vectors @ cv_vectors.T
    results = []

    for i, req in enumerate(requirements):
        # Combine semantic relevance and lexical overlap to rank evidence.
        ranking = []
        for j, sentence in enumerate(sentences):
            sim = float(sims[i, j])
            overlap = lexical_overlap(req["Criterion"], sentence)
            combined = 0.78 * sim + 0.22 * overlap
            ranking.append((combined, sim, overlap, j))
        ranking.sort(reverse=True)

        combined, sim, overlap, idx = ranking[0]
        excerpt = sentences[idx] if sim >= 0.34 else ""

        # Initial automated estimate; recruiters can override each decision.
        if sim >= 0.53 and (overlap >= 0.15 or sim >= 0.67):
            status, credit = "Strong evidence candidate", 1.0
        elif sim >= 0.39 and (overlap >= 0.10 or sim >= 0.48):
            status, credit = "Partial evidence candidate", 0.5
        else:
            status, credit = "Evidence not established", 0.0

        results.append({
            "Criterion": req["Criterion"],
            "Section": req["Section"],
            "Best CV evidence": excerpt,
            "Similarity": round(sim, 3),
            "Evidence status": status,
            "Recruiter decision": {
                1.0: "Supported",
                0.5: "Partial",
                0.0: "Not demonstrated",
            }[credit],
            "_initial_credit": credit,
        })

    guidance = {}
    for item in selected:
        section = item.get("section", "")
        if section not in guidance:
            guidance[section] = []
        guidance[section].append(item.get("text", ""))

    return results, guidance, sentences


def calculate_score(df):
    # Calculate each category independently, then apply the 70/30 weights.
    category_scores = {}
    category_details = {}

    mapping = {
        "Supported": 1.0,
        "Partial": 0.5,
        "Not demonstrated": 0.0,
    }

    for section, weight in WEIGHTS.items():
        subset = df[df["Section"] == section]
        if subset.empty:
            category_scores[section] = None
            category_details[section] = (0, 0)
            continue

        credits = subset["Recruiter decision"].map(mapping)
        category_scores[section] = float(credits.mean() * 100)
        category_details[section] = (
            int((credits > 0).sum()), len(subset)
        )

    available = [
        (WEIGHTS[name], score)
        for name, score in category_scores.items()
        if score is not None
    ]
    total_weight = sum(w for w, _ in available)
    overall = (
        sum(w * score for w, score in available) / total_weight
        if total_weight else 0.0
    )
    return overall, category_scores, category_details


def get_chat_context(question, role, level, kb, embedder, evaluation):
    selected = select_role_chunks(kb, role, level)
    if not selected:
        return "No knowledge-base passages found for this role and level."

    texts = [item.get("text", "") for item in selected]
    vectors = embedder.encode(
        texts, convert_to_numpy=True, normalize_embeddings=True
    ).astype(np.float32)
    q_vector = embedder.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True
    ).astype(np.float32)[0]
    scores = vectors @ q_vector




def generate_answer(question, role, level, kb, embedder, evaluation, history):
    tokenizer, model = load_chat_model()

    context_result = get_chat_context(
        question, role, level, kb, embedder, evaluation
    )

    # Handle either a plain string or a tuple result.
    if isinstance(context_result, tuple):
        context = context_result[0] if context_result else ""
    else:
        context = context_result

    context = context if isinstance(context, str) else ""
    question = question if isinstance(question, str) else ""
    history = history if isinstance(history, list) else []

    system_prompt = """
You are FALCON, an engineering career and CV assessment assistant.
Answer using the provided knowledge-base passages and CV assessment context.
Distinguish facts in the sources from your own general suggestions.
If the knowledge base does not contain an answer, say so clearly.
Never invent certifications, requirements, candidate achievements, or citations.
For skill gaps, suggest practical study topics and projects with measurable deliverables.
A similarity score is not proof of competence. A CV match score is not a probability
of being hired. Do not expose private information unnecessarily.
Keep answers clear, practical, and specific.
You cannot answer any question outside the dataset you have. This is prohibited.
"""

    messages = [{"role": "system", "content": system_prompt}]

    if context.strip():
        messages.append({
            "role": "user",
            "content": "Use this context for the conversation:\n" + context,
        })

    # Keep recent turns and sanitize their content.
    for message in history[-8:]:
        if not isinstance(message, dict):
            continue

        msg_role = message.get("role")
        msg_content = message.get("content")

        if msg_role in ("user", "assistant") and isinstance(msg_content, str):
            messages.append({
                "role": msg_role,
                "content": msg_content[:3000],
            })

    messages.append({"role": "user", "content": question})

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=7000,
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=450,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    return answer or "I couldn't generate an answer. Please try rephrasing your question."



# ---------- Sidebar ----------
kb = load_kb()

with st.sidebar:
    st.markdown("## 🦅 FALCON-GPT")
    st.caption("Understand your position for a better decision.")
    role = st.selectbox("Target role", ROLES)
    level = st.selectbox(
        "Career level",
        LEVELS,
        format_func=lambda x: {
            "junior": "Junior · 0–2 years",
            "mid_level": "Mid-level · 2–5 years",
            "senior": "Senior · 5+ years",
        }[x],
    )
    st.divider()
    st.caption("Evidence-first assessment")
    st.caption("Required skills: 70%")
    st.caption("Preferred skills: 30%")
    st.caption("Human review recommended")

st.markdown("""
<div class="hero">
<h1>FALCON-GPT</h1>
<p>See further. Aim higher.</p>
</div>
""", unsafe_allow_html=True)

assessment_tab, chat_tab = st.tabs([
    "📊 CV Assessment & Roadmap",
    "💬 Knowledge Base Chat",
])

with assessment_tab:
    st.markdown("### 1. Upload a candidate CV")
    uploaded = st.file_uploader(
        "PDF resume", type=["pdf"], key="cv_upload"
    )

    if st.button("Evaluate CV", type="primary", use_container_width=True):
        if uploaded is None:
            st.error("Upload a PDF CV first.")
        else:
            try:
                cv_text = extract_pdf(uploaded)
                with st.spinner("Loading embeddings and evaluating the CV..."):
                    embedder = load_embedding_model()
                    rows, guidance, sentences = evaluate_cv(
                        cv_text, role, level, kb, embedder
                    )
                st.session_state["current_assessment"] = {
                    "role": role,
                    "level": level,
                    "filename": uploaded.name,
                    "rows": rows,
                    "guidance": guidance,
                    "cv_text": cv_text,
                    "sentences": sentences,
                }
                st.success("CV assessment completed.")
            except Exception as e:
                st.error(f"Evaluation failed: {e}")

    assessment = st.session_state.get("current_assessment")

    if assessment:
        if assessment["role"] != role or assessment["level"] != level:
            st.info("Role or level changed. Run the evaluation again.")
        else:
            st.markdown("### 2. Role-match report")
            st.caption(
                f"{assessment['filename']} · {role} · "
                f"{level.replace('_', ' ').title()}"
            )

            df = pd.DataFrame(assessment["rows"])
            df["Recruiter decision"] = df["Recruiter decision"].replace({
                "Not demonstrated": "Not demonstrated"
            })

            score, section_scores, details = calculate_score(df)

            c1, c2, c3 = st.columns(3)
            with c1:
                st.markdown(
                    f'<div class="metric"><div class="metric-label">'
                    f'Estimated CV-to-role match</div><div class="metric-value">'
                    f'{score:.1f}%</div></div>', unsafe_allow_html=True
                )
            with c2:
                st.markdown(
                    f'<div class="metric"><div class="metric-label">'
                    f'Required skills</div><div class="metric-value">'
                    f'{section_scores.get("Required Skills", 0):.1f}%</div></div>',
                    unsafe_allow_html=True
                )
            with c3:
                st.markdown(
                    f'<div class="metric"><div class="metric-label">'
                    f'Preferred skills</div><div class="metric-value">'
                    f'{section_scores.get("Preferred Skills", 0):.1f}%</div></div>',
                    unsafe_allow_html=True
                )

            st.progress(int(round(score)))
            st.warning(
                "This is a preliminary CV evidence-match estimate, not a probability "
                "of being hired or a validated measure of real-world ability. "
                "Review the evidence and correct the recruiter decisions below."
            )

            st.markdown("### 3. Review evidence and adjust decisions")
            st.write(
                "The initial decisions are automated estimates. Change them where "
                "the evidence does not justify the suggested rating."
            )

            editable = df.drop(columns=["_initial_credit"], errors="ignore")
            edited = st.data_editor(
                editable,
                use_container_width=True,
                hide_index=True,
                key="assessment_editor",
                disabled=[
                    "Criterion", "Section", "Best CV evidence",
                    "Similarity", "Evidence status",
                ],
                column_config={
                    "Similarity": st.column_config.NumberColumn(
                        "Similarity (not a confidence score)", format="%.3f"
                    ),
                    "Recruiter decision": st.column_config.SelectboxColumn(
                        options=["Supported", "Partial", "Not demonstrated"],
                        required=True,
                    ),
                },
            )

            updated_score, updated_sections, updated_details = calculate_score(edited)
            st.markdown("### Updated score from your decisions")
            st.metric("Reviewed CV-to-role match", f"{updated_score:.1f}%")
            st.write(
                f"Required skills: {updated_sections.get('Required Skills', 0):.1f}% · "
                f"Preferred skills: {updated_sections.get('Preferred Skills', 0):.1f}%"
            )

            st.download_button(
                "Download assessment CSV",
                data=edited.to_csv(index=False).encode("utf-8-sig"),
                file_name="talentlens_assessment.csv",
                mime="text/csv",
            )

            weak = edited[
                edited["Recruiter decision"].isin(["Partial", "Not demonstrated"])
            ]
            st.markdown("### 4. Your weak points and development roadmap")
            if weak.empty:
                st.success(
                    "No weak points are currently marked. Check that every "
                    "automated decision is justified by the CV evidence."
                )
            else:
                for _, row in weak.iterrows():
                    with st.expander(
                        f"{row['Section']}: {row['Criterion']}"
                    ):
                        st.write("**Why it needs attention:**")
                        if row["Best CV evidence"]:
                            st.write(
                                "The CV contains a possible related statement, "
                                "but the selected decision indicates that it does "
                                "not fully establish this requirement."
                            )
                            st.write("**Retrieved CV statement:**")
                            st.write(row["Best CV evidence"])
                        else:
                            st.write(
                                "No sufficiently relevant CV statement was retrieved. "
                                "This does not prove the candidate lacks the skill."
                            )
                        st.write("**Study action:**")
                        st.write(
                            f"Study the fundamentals of {row['Criterion']}, then "
                            "practice them in a small, testable implementation."
                        )
                        st.write("**Project action:**")
                        st.write(
                            "Build a project that demonstrates this skill, document "
                            "the implementation choices, and include test results "
                            "or measurable outcomes in the CV."
                        )

            st.markdown("### 5. Role-specific knowledge-base resources")
            guidance = assessment["guidance"]
            for key in ("education_experience", "certifications", "projects"):
                if key in guidance:
                    with st.expander(SECTION_NAMES.get(key, key)):
                        st.write("\n\n".join(guidance[key]))

            # Save the reviewed state for use in the chat tab.
            st.session_state["chat_assessment_summary"] = (
                f"Role: {role}; level: {level}; estimated match: "
                f"{updated_score:.1f}%; required skills: "
                f"{updated_sections.get('Required Skills', 0):.1f}%; "
                f"preferred skills: "
                f"{updated_sections.get('Preferred Skills', 0):.1f}%. "
                f"Reviewed requirements: {len(edited)}. "
                "These are CV evidence estimates, not hiring probabilities."
            )

with chat_tab:
    st.markdown("### Ask questions about the knowledge base")
    st.write(
        "Ask about role requirements, skill gaps, projects, certifications, "
        "career levels, or the current CV assessment. Answers use retrieved "
        "knowledge-base passages as context."
    )

    if "chat_history" not in st.session_state:
        st.session_state["chat_history"] = []

    for msg in st.session_state["chat_history"]:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])

    question = st.chat_input(
        "Example: What projects should a junior robotics engineer build?"
    )

    if question:
        history = st.session_state["chat_history"]
        history.append({"role": "user", "content": question})

        with st.chat_message("user"):
            st.markdown(question)

        with st.chat_message("assistant"):
            try:
                embedder = load_embedding_model()
                assessment_context = None
                if st.session_state.get("chat_assessment_summary"):
                    assessment_context = {
                        "summary": st.session_state["chat_assessment_summary"]
                    }
                with st.spinner("Retrieving context and generating an answer..."):
                    answer = generate_answer(
                        question, role, level, kb, embedder,
                        assessment_context, history[:-1]
                    )
                st.markdown(answer)
                history.append({"role": "assistant", "content": answer})
                st.session_state["chat_history"] = history
            except Exception as e:
                st.error(
                    f"Chat model could not answer: {e}. "
                    "Check internet access, model download status, and available RAM/GPU."
                )

    if st.button("Clear chat history"):
        st.session_state["chat_history"] = []
        st.rerun()

st.divider()
st.caption(
    "FALCON prototype · Review automated matches before using them in recruitment. "
    "Do not treat missing CV evidence as proof of missing ability."
)
'''

with open(APP_DIR / "app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("Updated app.py:", APP_DIR / "app.py")
print("Knowledge base preserved:", (APP_DIR / "knowledge_base.json").exists())


Updated app.py: /kaggle/working/cv_evaluator/app.py
Knowledge base preserved: True


In [905]:
from pathlib import Path
import re

app_path = Path("/kaggle/working/cv_evaluator/app.py")
source = app_path.read_text(encoding="utf-8")

# 1. Stop if the tab may already have been added.
if re.search(r"with\s+manual_tab\s*:", source):
    raise ValueError(
        "A Manual Profile tab already exists. "
        "Inspect app.py before trying to insert it again."
    )

# 2. Find the two-tab declaration despite indentation/spacing differences.
tabs_pattern = re.compile(
    r"(?m)^[ \t]*"
    r"(?P<assessment>\w+)\s*,\s*(?P<chat>\w+)"
    r"\s*=\s*st\.tabs\(\s*\["
    r"(?P<body>.*?)"
    r"\]\s*\)",
    re.DOTALL,
)

match = tabs_pattern.search(source)

if not match:
    print("Could not identify the tab declaration.")
    print("\nActual st.tabs declarations in app.py:")

    for line_number, line in enumerate(source.splitlines(), start=1):
        if "st.tabs(" in line:
            print(f"{line_number}: {line.strip()}")

    raise ValueError(
        "No changes were made. Use the printed declaration to adapt the script."
    )

assessment_tab = match.group("assessment")
chat_tab = match.group("chat")
tabs_body = match.group("body")

# Confirm the expected two original tabs are present.
if "CV Assessment" not in tabs_body or "Knowledge Base Chat" not in tabs_body:
    print("Found tab declaration, but its labels differ:")
    print(tabs_body)
    raise ValueError("No changes were made because the tab labels differ.")

# 3. Find the corresponding chat section.
chat_section_pattern = re.compile(
    rf"(?m)^with\s+{re.escape(chat_tab)}\s*:\s*$"
)

chat_match = chat_section_pattern.search(source)

if not chat_match:
    raise ValueError(
        f"Found tab variable '{chat_tab}', but could not find "
        f"'with {chat_tab}:'. No changes were made."
    )

# 4. Prepare a three-tab declaration.
new_tabs = f'''{assessment_tab}, manual_tab, {chat_tab} = st.tabs([
    "📊 CV Assessment & Roadmap",
    "✍️ Manual Profile",
    "💬 Knowledge Base Chat",
])'''

# 5. Inject the existing manual UI before the chat section.
# manual_ui must already contain your complete Manual Profile code.
if "manual_ui" not in globals():
    raise NameError(
        "manual_ui is not defined. Run the cell that defines manual_ui first."
    )

updated_source = (
    source[:match.start()]
    + new_tabs
    + source[match.end():chat_match.start()]
    + "\n"
    + manual_ui.strip("\n")
    + "\n\n"
    + source[chat_match.start():]
)

# 6. Compile before writing so syntax errors don't overwrite app.py.
compile(updated_source, str(app_path), "exec")

app_path.write_text(updated_source, encoding="utf-8")

print("Manual Profile tab added successfully.")
print("Updated:", app_path)
print("Tabs:", assessment_tab, "manual_tab", chat_tab)

Manual Profile tab added successfully.
Updated: /kaggle/working/cv_evaluator/app.py
Tabs: assessment_tab manual_tab chat_tab


In [907]:
from pathlib import Path

app_path = Path("/kaggle/working/cv_evaluator/app.py")
source = app_path.read_text(encoding="utf-8")

start_marker = '<style>'
end_marker = '</style>'

start = source.find(start_marker)
end = source.find(end_marker, start)

if start == -1 or end == -1:
    raise ValueError("Could not locate the existing CSS style block.")

end += len(end_marker)

new_css = r"""<style>
/* ---------- FALCON design system ---------- */
:root {
  --tl-blue: #3978f6;
  --tl-blue-hover: #2862d7;
  --tl-teal: #18a999;
  --tl-radius: 16px;
}

/* Respect Streamlit's active light/dark theme */
.stApp {
  background: var(--background-color);
  color: var(--text-color);
}

[data-testid="stHeader"] {
  background: transparent;
}

[data-testid="stMainBlockContainer"] {
  padding-top: 2rem;
  padding-bottom: 3rem;
  max-width: 1440px;
}

/* Sidebar: use theme-aware text instead of forcing every child white */
[data-testid="stSidebar"] {
  border-right: 1px solid var(--border-color);
}

[data-testid="stSidebar"] label,
[data-testid="stSidebar"] p,
[data-testid="stSidebar"] .stMarkdown,
[data-testid="stSidebar"] [data-testid="stWidgetLabel"] {
  color: var(--text-color) !important;
}

/* Dropdowns: readable labels, selected values, and menus */
div[data-baseweb="select"] > div {
  border-radius: 10px;
  border-color: var(--border-color);
}

div[data-baseweb="select"] input,
div[data-baseweb="select"] span,
div[data-baseweb="select"] div {
  color: var(--text-color);
}

div[data-baseweb="popover"],
ul[role="listbox"],
li[role="option"] {
  color: var(--text-color);
}

li[role="option"] {
  background: var(--secondary-background-color);
}

li[role="option"]:hover {
  background: var(--primary-color);
}

/* Hero banner */
.tl-hero {
  padding: 2.2rem 2.3rem;
  border-radius: 22px;
  margin-bottom: 1.5rem;
  color: #ffffff !important;
  background:
    radial-gradient(circle at 88% 15%, rgba(73, 221, 204, .25), transparent 28%),
    linear-gradient(125deg, #142b4d 0%, #205da8 62%, #247e91 100%);
  box-shadow: 0 12px 35px rgba(25, 75, 130, .15);
}

.tl-hero h1,
.tl-hero h2,
.tl-hero p,
.tl-hero span {
  color: #ffffff !important;
}

.tl-hero p {
  font-size: 1.03rem;
  line-height: 1.7;
  opacity: .92;
}

/* Section headings and helpful text */
h1, h2, h3, h4 {
  letter-spacing: -0.025em;
}

[data-testid="stCaptionContainer"] {
  opacity: .85;
}

div[data-testid="stAlert"] {
  border-radius: 12px;
}

/* Tabs */
button[data-baseweb="tab"] {
  font-weight: 650;
  padding-top: 12px;
  padding-bottom: 12px;
}

button[data-baseweb="tab"] p {
  font-size: .98rem;
}

div[data-baseweb="tab-highlight"] {
  background-color: var(--primary-color);
}

/* Inputs and text areas */
div[data-testid="stTextInput"] input,
div[data-testid="stTextArea"] textarea,
div[data-testid="stNumberInput"] input {
  border-radius: 10px;
}

div[data-testid="stTextArea"] textarea {
  line-height: 1.55;
}

/* Buttons */
div.stButton > button,
div[data-testid="stFormSubmitButton"] button,
div[data-testid="stDownloadButton"] button {
  border-radius: 10px;
  min-height: 43px;
  font-weight: 600;
  transition: border-color .15s ease, transform .15s ease;
}

div.stButton > button:hover,
div[data-testid="stFormSubmitButton"] button:hover,
div[data-testid="stDownloadButton"] button:hover {
  border-color: var(--primary-color);
  transform: translateY(-1px);
}

div.stButton > button[kind="primary"],
div[data-testid="stFormSubmitButton"] button[kind="primary"] {
  background: var(--primary-color);
  color: #ffffff;
  border: 0;
}

/* Metrics: no fixed white cards, so text remains legible in both themes */
[data-testid="stMetric"] {
  padding: 16px 18px;
  border: 1px solid var(--border-color);
  border-radius: var(--tl-radius);
  background: var(--secondary-background-color);
}

[data-testid="stMetricLabel"] {
  color: var(--text-color);
  opacity: .8;
}

[data-testid="stMetricValue"] {
  color: var(--text-color);
  font-weight: 750;
}

/* Tables and editable evidence */
[data-testid="stDataFrame"],
[data-testid="stDataEditor"] {
  border: 1px solid var(--border-color);
  border-radius: 12px;
  overflow: hidden;
}

[data-testid="stDataFrame"] *,
[data-testid="stDataEditor"] * {
  color: var(--text-color);
}

/* Expanders and forms */
[data-testid="stExpander"] {
  border: 1px solid var(--border-color);
  border-radius: 12px;
  overflow: hidden;
}

[data-testid="stForm"] {
  border: 1px solid var(--border-color);
  border-radius: 16px;
  padding: 1.25rem;
}

/* Progress bar */
div[data-testid="stProgress"] > div > div {
  background-color: var(--primary-color);
  border-radius: 99px;
}

/* Mobile spacing */
@media (max-width: 768px) {
  [data-testid="stMainBlockContainer"] {
    padding: 1rem 1rem 2rem;
  }

  .tl-hero {
    padding: 1.4rem;
    border-radius: 16px;
  }
}
</style>"""

source = source[:start] + new_css + source[end:]
app_path.write_text(source, encoding="utf-8")

print("Updated FALCON theme-aware styling.")
print("Next: restart the Streamlit app using your existing launch cell.")

Updated FALCON theme-aware styling.
Next: restart the Streamlit app using your existing launch cell.


In [908]:

import getpass
import subprocess
import time
import requests
from pyngrok import ngrok

APP_DIR = "/kaggle/working/cv_evaluator"

# Stop the previous Streamlit process if it exists in this notebook session.
try:
    streamlit_process.terminate()
    streamlit_process.wait(timeout=5)
except Exception:
    pass

# Disconnect the previous ngrok tunnel.
try:
    for tunnel in ngrok.get_tunnels():
        ngrok.disconnect(tunnel.public_url)
except Exception:
    pass

# Configure ngrok. Your token is entered privately.
token = getpass.getpass("Insert your AuthToken Here:")
ngrok.set_auth_token(token)

# Start the updated website.
log_file = open("/kaggle/working/streamlit.log", "w")
streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", f"{APP_DIR}/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--browser.gatherUsageStats", "false",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

# Wait for Streamlit to become available.
ready = False
for _ in range(60):
    try:
        response = requests.get(
            "http://127.0.0.1:8501/_stcore/health", timeout=2
        )
        if response.status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(1)

if ready:
    tunnel = ngrok.connect(8501, "http")
    print("FALCON is running!")
    print("Open this URL:", tunnel.public_url)
    print("Keep this Kaggle session running.")
else:
    print("Streamlit did not start. Recent log output:")
    with open("/kaggle/working/streamlit.log", encoding="utf-8") as f:
        print(f.read()[-6000:])


t=2026-10-11T18:37:10+0000 lvl=warn msg="failed to open private leg" id=f5c0c69e9410 privaddr=localhost:8501 err="dial tcp [::1]:8501: connect: connection refused"
t=2026-10-11T18:37:10+0000 lvl=warn msg="failed to open private leg" id=978c30541dd1 privaddr=localhost:8501 err="dial tcp [::1]:8501: connect: connection refused"
t=2026-10-11T18:37:11+0000 lvl=warn msg="failed to open private leg" id=923c109ab1d5 privaddr=localhost:8501 err="dial tcp [::1]:8501: connect: connection refused"
t=2026-10-11T18:37:11+0000 lvl=warn msg="failed to open private leg" id=fb1bcefedb98 privaddr=localhost:8501 err="dial tcp [::1]:8501: connect: connection refused"
t=2026-10-11T18:37:12+0000 lvl=warn msg="failed to open private leg" id=59fce0e9c3ef privaddr=localhost:8501 err="dial tcp [::1]:8501: connect: connection refused"
t=2026-10-11T18:37:12+0000 lvl=warn msg="failed to open private leg" id=9847517d0fcf privaddr=localhost:8501 err="dial tcp [::1]:8501: connect: connection refused"
t=2026-10-11T18:

Insert your AuthToken Here: ········


FALCON is running!
Open this URL: https://hardening-attach-jet.ngrok-free.dev
Keep this Kaggle session running.
